# Analysis

**Hypothesis**: Spatial organization of each cardiac population’s neighborhood composition (i.e., which other populations surround it) systematically varies with disease-relevant 'Complexity' and 'Purity', revealing microenvironmental remodeling that is not captured by cell-intrinsic maturation signatures alone.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_anon.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Spatial organization of each cardiac population’s neighborhood composition (i.e., which other populations surround it) systematically varies with disease-relevant 'Complexity' and 'Purity', revealing microenvironmental remodeling that is not captured by cell-intrinsic maturation signatures alone.

## Steps:
- Inspect and clean metadata (Populations, Sample_ID, Batch, Complexity, Purity), define global Complexity/Purity tertiles, and summarize for each Population its cell count, per-tertile coverage, distributional statistics (mean/SD and quantiles) of Complexity/Purity, and per-Sample_ID contribution to select well-powered populations spanning multiple tertiles.
- For well-powered Populations, restrict to cells with complete metadata, optionally subset within each Sample_ID, and construct a spatial k-nearest-neighbor graph in physical space from `.obsm['spatial']` (with a primary k, e.g. k=10), computing for each cell the local neighborhood composition as the fraction of neighbors belonging to each Population.
- Aggregate per-cell neighborhood compositions by focal Population, Complexity tertile (and secondarily Purity tertile), and Sample_ID to obtain, for each focal Population and tertile, the mean and variance of neighboring-Population frequencies, focusing on key cardiac Populations (e.g., PA–PZ) with sufficient cells across tertiles and samples.
- Within each focal Population, test whether neighborhood composition differs between low- and high-Complexity tertiles (and separately low- vs high-Purity) using per-neighbor-population univariate tests (e.g., Wilcoxon rank-sum on neighbor fractions) with Benjamini–Hochberg correction, reporting effect sizes and adjusted p-values for each neighboring Population.
- Identify and report focal Populations and neighbor-Population pairs that show the strongest Complexity-associated shifts in spatial microenvironment, and examine consistency across Sample_ID by repeating tests within each sample (where powered) and comparing effect directions and magnitudes.
- As a robustness check, repeat the neighborhood analysis with alternative k values (e.g., k=6 and k=20) and quantify stability of key Complexity-associated shifts using sign concordance and rank correlation of effect sizes across k settings.


## This code cleans and inspects cell-level metadata in an AnnData object, defines global tertiles for Complexity and Purity, and summarizes these metrics per cell population to identify “well-powered” populations for downstream spatial neighborhood analyses. It ensures consistent, complete metadata, annotates each cell with Complexity/Purity tertiles, and then computes per-population statistics and coverage thresholds to select populations with sufficient cells, samples, and tertile representation.

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Inspect, clean metadata, and define tertiles for Complexity and Purity

print('AnnData shape (cells x genes):', adata.shape)
print('\n.obs columns:', list(adata.obs.columns))
print('\n.obsm keys:', list(adata.obsm.keys()))

# Ensure required metadata columns exist
required_cols = ['Populations', 'Complexity', 'Purity', 'Sample_ID']
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    print('Missing required columns in adata.obs:', missing)
else:
    print('All required metadata columns found.')

# Work on a copy of obs
obs = adata.obs.copy()

# Coerce Complexity and Purity to numeric if needed
for col in ['Complexity', 'Purity']:
    if not np.issubdtype(obs[col].dtype, np.number):
        obs[col] = pd.to_numeric(obs[col], errors='coerce')

# Track missingness before dropping
initial_n = obs.shape[0]
missing_mask = obs[required_cols].isna().any(axis=1)
print(f"\nTotal cells with any missing required metadata: {missing_mask.sum()} ({missing_mask.mean():.3%} of all cells)")

# Drop cells with missing key metadata for downstream consistency
obs_clean = obs.loc[~missing_mask].copy()
print(f"Retained {obs_clean.shape[0]} cells ({obs_clean.shape[0] / initial_n:.3%}) after filtering for complete metadata.")

# Record which cells are retained for downstream analyses
valid_idx = obs_clean.index
adata.uns['spatial_neighbor_valid_cells'] = valid_idx.to_list()

# Define global tertiles for Complexity and Purity on the cleaned set
for col in ['Complexity', 'Purity']:
    q = obs_clean[col].quantile([0.33, 0.67]).values
    low_thr, high_thr = q[0], q[1]
    tertile_col = f'{col}_tertile'
    obs_clean[tertile_col] = pd.cut(
        obs_clean[col],
        bins=[-np.inf, low_thr, high_thr, np.inf],
        labels=['low', 'mid', 'high']
    )
    print(f"\n{col} tertile cutpoints (low, high): {low_thr:.3f}, {high_thr:.3f}")

# Attach tertile labels back to adata.obs for reuse
for col in ['Complexity_tertile', 'Purity_tertile']:
    ser = pd.Series(index=obs_clean.index, data=obs_clean[col]).reindex(adata.obs.index)
    adata.obs[col] = pd.Categorical(ser)

# Per-population missingness summary (fraction dropped)
missing_summary = (
    obs.assign(missing_any=missing_mask)
       .groupby('Populations')['missing_any']
       .agg(n_total='count', n_missing='sum')
       .reset_index()
)
missing_summary['frac_missing'] = missing_summary['n_missing'] / missing_summary['n_total']

print('\nPer-population fraction of cells dropped due to missing metadata (top 25 by n_total):')
print(missing_summary.sort_values('n_total', ascending=False).head(25).to_string(index=False))

# Summarize populations on the cleaned set
pop_group = obs_clean.groupby('Populations')
summary_list = []
for pop, df in pop_group:
    n_cells = df.shape[0]
    n_samples = df['Sample_ID'].nunique()
    # Complexity/Purity distribution stats
    complexity_mean = df['Complexity'].mean()
    complexity_sd = df['Complexity'].std()
    complexity_q10, complexity_median, complexity_q90 = df['Complexity'].quantile([0.1, 0.5, 0.9]).values
    purity_mean = df['Purity'].mean()
    purity_sd = df['Purity'].std()
    purity_q10, purity_median, purity_q90 = df['Purity'].quantile([0.1, 0.5, 0.9]).values
    # Tertile coverage counts
    compl_counts = df['Complexity_tertile'].value_counts().reindex(['low', 'mid', 'high'], fill_value=0)
    purity_counts = df['Purity_tertile'].value_counts().reindex(['low', 'mid', 'high'], fill_value=0)
    summary_list.append({
        'Population': pop,
        'n_cells': n_cells,
        'n_samples': n_samples,
        'Complexity_mean': complexity_mean,
        'Complexity_sd': complexity_sd,
        'Complexity_q10': complexity_q10,
        'Complexity_median': complexity_median,
        'Complexity_q90': complexity_q90,
        'Purity_mean': purity_mean,
        'Purity_sd': purity_sd,
        'Purity_q10': purity_q10,
        'Purity_median': purity_median,
        'Purity_q90': purity_q90,
        'Complexity_low_n': compl_counts.loc['low'],
        'Complexity_mid_n': compl_counts.loc['mid'],
        'Complexity_high_n': compl_counts.loc['high'],
        'Purity_low_n': purity_counts.loc['low'],
        'Purity_mid_n': purity_counts.loc['mid'],
        'Purity_high_n': purity_counts.loc['high'],
    })

summary_df = pd.DataFrame(summary_list)

# Heuristic for "well-powered" populations: sufficient cells, samples, and coverage across Complexity tertiles
min_cells = 1000
min_samples = 2
min_cells_per_tertile = 100
summary_df['has_two_complexity_tertiles'] = (
    (summary_df[['Complexity_low_n', 'Complexity_mid_n', 'Complexity_high_n']] >= min_cells_per_tertile).sum(axis=1) >= 2
)
summary_df['well_powered'] = (
    (summary_df['n_cells'] >= min_cells) &
    (summary_df['n_samples'] >= min_samples) &
    summary_df['has_two_complexity_tertiles']
)

summary_df = summary_df.sort_values('n_cells', ascending=False)

print('\nPopulation summary with Complexity/Purity distribution stats (top 25 by cell count):')
print(summary_df.head(25).to_string(index=False))

print('\nWell-powered populations for downstream spatial neighborhood analysis (meeting cell, sample, and Complexity-tertile coverage thresholds):')
print(summary_df[summary_df['well_powered']].to_string(index=False))

# Store summary in adata.uns for downstream reuse
adata.uns['population_complexity_purity_summary'] = summary_df.to_dict(orient='list')

# Also print a convenient Python list of well-powered population names
well_powered_pops = summary_df.loc[summary_df['well_powered'], 'Population'].tolist()
print('\nWell-powered population names:')
print(well_powered_pops)

AnnData shape (cells x genes): (228635, 238)

.obs columns: ['Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity']

.obsm keys: ['X_umap', 'spatial']
All required metadata columns found.

Total cells with any missing required metadata: 0 (0.000% of all cells)
Retained 228635 cells (100.000%) after filtering for complete metadata.

Complexity tertile cutpoints (low, high): 9.000, 11.000

Purity tertile cutpoints (low, high): 0.427, 0.557

Per-population fraction of cells dropped due to missing metadata (top 25 by n_total):
Populations  n_total  n_missing  frac_missing
         PA    30380          0           0.0
         PB    19947          0           0.0
         PC    17584          0           0.0
         PD    16624          0           0.0
         PE    16511          0           0.0
         PF    12248          0           0.0
         PG    11596          0           0.0
         PH    10887          0           0.0
         PI    10441        

### Agent Interpretation

You’re in a very strong position to tackle the neighborhood-structure hypothesis.

Key points from this step

1. **Metadata quality and coverage**
   - No missingness in `Populations`, `Complexity`, `Purity`, or `Sample_ID` (all 228,635 cells retained). This removes a major source of bias in neighborhood estimation and tertile comparisons.
   - Both `Complexity` and `Purity` are reasonably well-behaved numerically; global tertile cutpoints are:
     - Complexity: low ≤ 9, mid (9–11), high ≥ 11
     - Purity: low ≤ 0.427, mid (0.427–0.557), high ≥ 0.557

2. **Population-level structure relative to Complexity/Purity**
   - Many populations have **broad Complexity distributions** spanning low, mid, and high tertiles, which is ideal for within-population contrasts.
   - However, a subset of populations is **highly skewed** to one side of Complexity or Purity:
     - Example: `PB` has 17,523 low-Complexity cells vs only 717 high; `PS` is even more extreme (4,224 low vs 102 high).
     - Several populations are strongly biased to **high Purity** (e.g., `PB`, `PG`, `PI`, `PM`, `PR`, `PS`), while others are comparatively low-Purity (`PC`, `PD`, `PL`, `PJ`).
   - This “population–Complexity/Purity coupling” is important: it means some cell types are intrinsically more mature/less noisy or more pure. For the hypothesis, you’ll want to distinguish:
     - **Within-population changes across tertiles** (microenvironment remodeling at fixed cell type)
     - From **between-population differences in their typical Complexity/Purity levels** (intrinsic differences, not microenvironment.)

3. **Well-powered populations**
   - The heuristic is stringent but appropriate for neighborhood analysis: `min_cells = 1000`, `min_cells_per_tertile = 100`, `min_samples = 2`, and at least two Complexity tertiles.
   - Result: 27 populations (PA–PAA range) are labeled `well_powered`. These include all the big clusters (PA–PT, etc.) and also smaller-but-acceptable ones (PX, PY, PZ, PAA).
   - Many of these span all three Complexity tertiles with ≥100 cells in at least two tertiles, so **you can perform robust within-population low vs high comparisons** for most of them.

How this informs the hypothesis

Your hypothesis is that **spatial neighborhood composition for a given Population changes systematically with Complexity and Purity** in ways not explained solely by cell-intrinsic maturation signatures.

This step shows:

- You have **ample cell numbers and tertile coverage** across many populations; so:
  - You can design analyses where you **hold cell identity fixed** (e.g., only `PA`) and **contrast low vs high Complexity** within that population, which is exactly what you need to test microenvironment shifts.
- At the same time, the **population-specific distributions of Complexity and Purity** are quite heterogeneous:
  - Some populations live mostly in “low Complexity” states (e.g., PB, PS), others in “high Complexity” (e.g., PK, PN).
  - This is both a **signal** (different maturation states by population) and a **confounder** (global comparisons across populations and tertiles can mix cell-intrinsic and microenvironmental effects).

So far, nothing contradicts the hypothesis. The data structure is compatible with the idea that there could be **population-specific, disease-relevant microenvironment remodeling**, but we haven’t examined spatial neighborhoods yet. The promising sign is that many populations (including those spanning PA–PZ) are large and cross tertiles, so you will have statistical power to detect even moderate shifts.

Concrete suggestions for next steps and refinements

1. **Refine “well-powered” selection in light of skewed tertiles**
   - Before building spatial KNNs, identify **focal populations where both low and high Complexity tertiles are reasonably balanced**, for cleaner interpretation:
     - e.g., `PA`, `PC`, `PD`, `PE`, `PF`, `PH`, `PQ`, `PT`, `PV`, `PW`, `PZ`, `PAA` appear more balanced across Complexity tertiles.
     - For heavily skewed populations (e.g., `PB`, `PS`, `PR`), you can still analyze them, but:
       - Emphasize **effect size** over p-value (power imbalance).
       - Consider downsampling the dominant tertile to match the minority tertile per `Sample_ID` when doing within-population tests, to reduce confounding by sample and density.

2. **Prepare for spatial neighborhood computation (upcoming steps)**
   - You have `.obsm['spatial']` for all retained cells. For the next step (constructing the spatial graph), I recommend:
     - Build a single **global KNN in spatial coordinates** (e.g., k = 10 as planned) on the same filtered index `adata.uns['spatial_neighbor_valid_cells']` to keep consistency.
     - Store neighborhood composition as a matrix in `adata.obsm['neighbor_population_fractions_k10']` (cells × populations), with clear documentation.
   - Pay attention to **edge cells** and **section boundaries**:
     - Because sections/samples are disjoint in space, you should build KNNs **within each `Sample_ID` separately** rather than globally, to avoid meaningless neighbors across sections.
     - This will also help you when later examining consistency across samples.

3. **Control for sample structure and Complexity–Purity coupling**
   When you aggregate neighborhood composition by Complexity tertile within a focal Population, be careful of:
   - **Sample imbalance across tertiles**: Some tertiles may come disproportionately from certain `Sample_ID`s. Before or alongside per-tertile aggregation:
     - Compute per-sample counts of low/mid/high Complexity within each Population.
     - If severely imbalanced, consider:
       - Stratified analyses by `Sample_ID` (run tests within each sample and meta-analyze effect directions).
       - Or include `Sample_ID` as a blocking factor / random effect if you consider more advanced models later.
   - **Purity as a covariate**: Because Complexity and Purity are not independent, you’ll want to:
     - For the primary hypothesis, test **Complexity tertile effects within narrow Purity strata**, or
     - At minimum, perform **a secondary analysis within each Purity tertile** to see if Complexity-associated neighborhood shifts remain after roughly matching Purity.

4. **Select focal populations a priori for clearer biological stories**
   - To maintain independence from the paper and your prior Analysis 1, you might:
     - Focus on a **diverse subset of populations** across the Complexity and Purity spectrum:
       - E.g., high-complexity-biased (`PK`, `PN`), low-complexity-biased (`PB`, `PS`), and intermediate (`PA`, `PC`, `PD`, `PQ`), plus one or two smaller but interesting populations (`PZ`, `PAA`).
     - For each, ask: does the fraction of specific neighboring populations (e.g., PA vs PB vs PC, etc.) shift between low vs high Complexity?

5. **Define effect metrics that directly address the hypothesis**
   For step 4 in your plan (testing neighborhood composition differences):
   - For each focal Population F and each neighbor Population N:
     - Compute per-cell neighbor fraction `f_i(F→N)` (fraction of F’s neighbors that are N).
     - Compare distributions of `f_i(F→N)` between low and high Complexity cells in F, either:
       - Pooled across samples but with **sample-level covariate or stratification**, or
       - **Per-sample** and then meta-analyzed (e.g. sign test on effect direction).
   - Effect sizes to report:
     - Δ mean neighbor fraction (high – low).
     - Rank-based effect (e.g., Cliff’s delta or logit-transformed mean fraction difference).
   - This will directly tell you “in high-Complexity PA, the local neighborhood is X% more enriched for PB and Y% depleted for PC,” which is exactly the microenvironment remodeling signal.

6. **Use the summary stats to pre-empt confounding**
   You already computed per-population mean and quantiles for Complexity/Purity. Use them to:
   - Flag pairs of populations where **both Complexity and Purity distributions are similar**, so any neighborhood differences across tertiles cannot easily be attributed to drastically different intrinsic maturation levels.
   - Conversely, if a focal population’s Complexity tertile is strongly associated with being co-located with another intrinsically high-Complexity population, that may represent **macro-anatomical organization** rather than fine-grained microenvironment.

7. **Plan how to interpret results in a way that’s distinct from previous work**
   - Prior Analysis 1 focused on **gene expression signatures** of Complexity within populations.
   - This analysis centers on **spatial context**:
     - Focus interpretation on how **cell neighborhoods reconfigure** along Complexity/Purity:
       - e.g., “High-Complexity PA cells are more likely to be adjacent to PK and less to PB, suggesting migration/remodeling of their niche.”
     - You can explicitly check whether cells with “high-intrinsic-maturation signatures” (from Analysis 1, if you’ve defined them) also show **consistent microenvironment changes**, or if microenvironment effects are sometimes decoupled from intrinsic maturation signatures. That would directly support the hypothesis that microenvironment remodeling is not captured by cell-intrinsic signatures alone.

8. **Anticipate robustness checks**
   - Your plan to vary k (6, 10, 20) is good. Given the large cell numbers, you can easily:
     - Recompute neighborhood fractions for each k.
     - Assess **sign concordance** of Δ neighbor fractions across k values.
   - In addition, use:
     - A **radius-based neighborhood** (if spatial scale is somewhat homogeneous) as a complementary robustness check, to ensure KNN-based findings are not purely artifacts of local density differences.

Summary of how this step positions you relative to the hypothesis

- You have **clean, complete metadata** and a robust set of **well-powered populations** with good tertile coverage.
- The diversity and skew in Complexity/Purity distributions across populations actually enhance the richness of the problem, but also demand **careful within-population, sample-aware comparisons** to attribute differences to microenvironment rather than intrinsic maturation or sample composition.
- The current results **support moving forward** with the next steps (spatial KNN, neighborhood composition, and tertile-based tests). They do not yet validate the hypothesis, but they show the dataset has exactly the structure needed to test it in a statistically and biologically meaningful way.

In the next step, I’d recommend:
1) building per-sample spatial KNNs on `spatial_neighbor_valid_cells`,  
2) computing neighbor population fractions, and  
3) summarizing these fractions by (Population, Complexity_tertile, Sample_ID) to see early patterns—especially for a few focal populations with balanced tertiles (e.g., PA, PC, PD, PQ, PZ)—before diving into formal testing and robustness analyses.

## Next Steps
Step 1: Compute per-cell spatial neighborhood composition features by building per-sample (within-Sample_ID) k-nearest-neighbor graphs on `.obsm['spatial']` (primary k=10) for cells with complete metadata, and derive for each cell the fraction of neighbors belonging to each Population as neighborhood-composition features.
Step 2: Aggregate per-cell neighborhood compositions by focal Population × Complexity/Purity tertile × Sample_ID to obtain summary statistics (e.g., mean, median, variance) of neighboring-Population fractions, and use these to identify well-powered focal Populations (balanced low vs high tertiles across samples) for formal testing.
Step 3: Within each well-powered focal Population, test whether neighborhood composition differs between low- and high-Complexity tertiles (and separately low- vs high-Purity) for each neighbor Population using per-sample-stratified or per-sample-reported Wilcoxon rank-sum tests on per-cell neighbor fractions, applying Benjamini–Hochberg FDR control within a coherent family of Complexity- and Purity-based tests and reporting effect sizes.
Step 4: Compare and summarize the consistency of significant Complexity- and Purity-associated neighborhood shifts across Sample_IDs for each focal–neighbor Population pair, emphasizing robust, recurrent microenvironment changes with concordant effect directions and magnitudes across samples.
Step 5: Assess robustness of key focal–neighbor findings to alternative neighborhood definitions by repeating neighborhood-composition computation and core tests for alternative k values (e.g., k=6 and k=20) and quantifying stability of effect sizes and significance via sign concordance and rank correlation across k settings.

## This code builds per-sample spatial k-nearest-neighbor graphs (k=10) using `.obsm['spatial']`, computes for each valid cell the fraction of neighbors belonging to each Population, and stores these labeled neighbor-composition features in `adata` along with QC summaries and explicit tracking of cells without effective neighbors.

In [ ]:

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Step 1: Build per-sample spatial KNN (k=10) and compute neighbor population fractions

# Retrieve the set of valid cells previously identified
valid_cells = adata.uns.get('spatial_neighbor_valid_cells', list(adata.obs.index))
valid_cells = pd.Index(valid_cells)

# Restrict to valid cells
obs_sub = adata.obs.loc[valid_cells]
spatial_sub = adata.obsm['spatial'][obs_sub.index.to_numpy(), :]

# Sanity check: ensure Sample_ID and Populations are present
if 'Sample_ID' not in obs_sub.columns or 'Populations' not in obs_sub.columns:
    raise ValueError('Required columns Sample_ID and Populations are missing from adata.obs')

# Identify all populations to define consistent columns
all_pops = obs_sub['Populations'].astype('category').cat.categories.tolist()

# Prepare an array to store neighbor fractions: cells x populations
neighbor_frac = pd.DataFrame(0.0, index=obs_sub.index, columns=all_pops, dtype=float)

k = 10

# Process each sample separately to avoid cross-section neighbors
for sample_id, idx_in_sample in obs_sub.groupby('Sample_ID').groups.items():
    idx_in_sample = pd.Index(idx_in_sample)
    coords = spatial_sub[obs_sub.index.get_indexer(idx_in_sample), :]

    if coords.shape[0] == 0:
        continue

    # If fewer cells than k+1 in the sample, adjust k_eff (+1 because self will be dropped)
    k_eff = min(k + 1, coords.shape[0])

    # Build KD-tree for spatial KNN within this Sample_ID
    tree = cKDTree(coords)
    dists, neigh_idx = tree.query(coords, k=k_eff)

    # Ensure neigh_idx is 2D and integer-typed
    neigh_idx = np.atleast_2d(neigh_idx).astype(int)

    # neigh_idx is (n_cells_sample, k_eff); first neighbor is the cell itself
    # Convert neighbor indices (within-sample) to global indices
    sample_cells = idx_in_sample.to_numpy()

    for i, cell in enumerate(sample_cells):
        # If k_eff == 1, then this cell has only itself; skip as it has no neighbors
        if k_eff <= 1:
            continue

        # Exclude self: neighbors from position 1 onward
        neigh_local = neigh_idx[i][1:].astype(int)
        neigh_cells = sample_cells[neigh_local]
        neigh_pops = obs_sub.loc[neigh_cells, 'Populations'].to_numpy()

        # Compute fractions of each population among neighbors
        counts = pd.value_counts(neigh_pops, sort=False)
        frac = counts / counts.sum()
        neighbor_frac.loc[cell, frac.index] = frac.values

# Identify cells with no effective neighbors (all-zero rows) for downstream filtering/QC
no_neighbor_mask = neighbor_frac.sum(axis=1) == 0
no_neighbor_cells = neighbor_frac.index[no_neighbor_mask].tolist()
adata.uns['no_spatial_neighbors_k10'] = no_neighbor_cells

# Store a labeled version of neighbor fractions for downstream aggregation/testing
adata.uns['neighbor_population_fractions_k10_df_index'] = neighbor_frac.index.to_list()
adata.uns['neighbor_population_fractions_k10'] = neighbor_frac.copy()

# Store neighbor fractions as a NumPy array in adata.obsm for convenience
# Rows align with adata.obs.index; columns correspond to populations in
# adata.uns['neighbor_population_fractions_k10_cols'].
adata.obsm['neighbor_population_fractions_k10'] = neighbor_frac.loc[adata.obs.index].fillna(0.0).to_numpy()

# Also store mapping from columns to populations for later interpretation
adata.uns['neighbor_population_fractions_k10_cols'] = all_pops

# Basic text summary: mean self-neighbor fraction per population (as a quick QC of spatial clustering)
qc_rows = []
for pop in all_pops:
    pop_cells = obs_sub.index[obs_sub['Populations'] == pop]
    if len(pop_cells) == 0:
        continue
    vals = neighbor_frac.loc[pop_cells, pop]
    qc_rows.append({
        'Population': pop,
        'n_cells_valid': len(pop_cells),
        'mean_self_neighbor_fraction': float(vals.mean()),
        'median_self_neighbor_fraction': float(vals.median())
    })

qc_df = pd.DataFrame(qc_rows).sort_values('n_cells_valid', ascending=False)

n_no_neighbors = int(no_neighbor_mask.sum())
frac_no_neighbors = n_no_neighbors / neighbor_frac.shape[0] if neighbor_frac.shape[0] > 0 else 0.0

print('Per-population mean and median fraction of same-Populations neighbors (k=10):')
print(qc_df.to_string(index=False))
print(f"\nNumber of cells with no effective spatial neighbors at k=10: {n_no_neighbors} ({frac_no_neighbors:.3%} of valid cells)")


IndexError: arrays used as indices must be of integer (or boolean) type

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach

## Next Steps
Step 1: Construct per-sample (within-Sample_ID) spatial k-nearest neighbors (k=10) on `.obsm['spatial']` for cells with complete metadata (using `spatial_neighbor_valid_cells`), compute for each cell the fraction of neighbors belonging to each Population as neighborhood-composition features stored in `adata.obsm`/`adata.uns`, and explicitly track cells lacking effective neighbors.
Step 2: Aggregate per-cell neighborhood compositions by focal Population × Complexity_tertile × Purity_tertile × Sample_ID to obtain summary statistics (mean, median, variance) of neighboring-Population fractions, and use operational criteria (minimum cells per tertile and samples represented) to define well-powered focal Populations with reasonably balanced low vs high tertiles for formal testing.
Step 3: Within each well-powered focal Population, test whether neighborhood composition differs between low- and high-Complexity tertiles and, separately, between low- and high-Purity tertiles for each neighbor Population using per-sample-stratified Wilcoxon rank-sum tests on per-cell neighbor fractions, applying Benjamini–Hochberg FDR correction within each focal Population and reporting both effect sizes (Δ mean neighbor fraction) and adjusted p-values, while checking that effects are not driven by a single Sample_ID.
Step 4: Summarize the consistency and magnitude of significant Complexity- and Purity-associated neighborhood shifts across Sample_IDs and neighbor Populations, and attribute variance in neighbor fractions to Complexity, Purity, and Sample_ID using simple between- vs within-group variance summaries based on numpy/scipy to highlight robust, directionally concordant microenvironment remodeling not explained by intrinsic maturation alone.

## This code robustly computes per-sample spatial k-nearest neighbors (k=10) on `.obsm['spatial']` for metadata-complete cells, derives for each cell the fraction of neighboring cells belonging to each Population, and stores these neighborhood-composition features and QC summaries in `adata` while explicitly tracking cells that lack effective spatial neighbors.

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Step 1 (revised): Build per-sample spatial KNN (k=10) and compute neighbor population fractions robustly

# Retrieve the set of valid cells previously identified (from Step 1 metadata cleaning)
valid_cells = adata.uns.get('spatial_neighbor_valid_cells', list(adata.obs.index))
valid_cells = pd.Index(valid_cells)

# Defensive intersection to ensure valid_cells is a subset of adata.obs.index
valid_cells = valid_cells.intersection(adata.obs.index)
if valid_cells.empty:
    raise ValueError("No valid cells available after intersecting with adata.obs.index.")

# Subset obs and spatial coordinates to valid cells
obs_sub = adata.obs.loc[valid_cells].copy()
if 'spatial' not in adata.obsm_keys():
    raise ValueError(".obsm['spatial'] is missing from adata; spatial KNN cannot be computed.")

# Align spatial matrix to obs_sub index explicitly
spatial_full = adata.obsm['spatial']
# Build a mapping from global cell index position to cell ID for safety
obs_index_to_pos = pd.Series(np.arange(adata.n_obs), index=adata.obs.index)
coords_idx = obs_index_to_pos.loc[obs_sub.index].to_numpy()
spatial_sub = spatial_full[coords_idx, :]

# Ensure required columns are present
for col in ['Sample_ID', 'Populations']:
    if col not in obs_sub.columns:
        raise ValueError(f"Required column '{col}' is missing from adata.obs")

# Standardize Populations as a categorical with fixed levels across the dataset
all_pops = obs_sub['Populations'].astype('category').cat.categories.tolist()
neighbor_frac = pd.DataFrame(0.0, index=obs_sub.index, columns=all_pops, dtype=float)

k = 10
no_neighbor_cells = []

# Process each Sample_ID separately to avoid cross-section neighbors
for sample_id, sample_idx in obs_sub.groupby('Sample_ID').groups.items():
    sample_idx = pd.Index(sample_idx)
    # Coordinates for cells in this sample (in the same order as sample_idx)
    sample_pos = obs_index_to_pos.loc[sample_idx].to_numpy()
    coords = spatial_full[sample_pos, :]

    n_in_sample = coords.shape[0]
    if n_in_sample == 0:
        continue

    # Effective k within this sample (+1 to include self, which we'll drop)
    k_eff = min(k + 1, n_in_sample)

    # For samples with only 1 cell, there are no neighbors to compute
    if k_eff <= 1:
        no_neighbor_cells.extend(sample_idx.tolist())
        continue

    # Build KD-tree and query neighbors
    tree = cKDTree(coords)
    dists, neigh_idx = tree.query(coords, k=k_eff)

    # Ensure 2D neighbor index array
    neigh_idx = np.atleast_2d(neigh_idx).astype(int)

    # Map local indices (0..n_in_sample-1) to cell IDs
    sample_cells = sample_idx.to_numpy()

    # Iterate over cells in this sample
    for i, cell in enumerate(sample_cells):
        # Local neighbor indices (including self at position 0)
        local_neighbors = neigh_idx[i]
        # Exclude self if present (assume first entry is self for cKDTree)
        if local_neighbors.shape[0] > 0:
            local_neighbors = local_neighbors[1:]

        # If no real neighbors remain, record and skip
        if local_neighbors.size == 0:
            no_neighbor_cells.append(cell)
            continue

        neigh_cells = sample_cells[local_neighbors]
        neigh_pops = obs_sub.loc[neigh_cells, 'Populations'].to_numpy()

        # Compute neighbor population fractions
        if neigh_pops.size == 0:
            no_neighbor_cells.append(cell)
            continue

        counts = pd.value_counts(neigh_pops, sort=False)
        total_neighbors = counts.sum()
        if total_neighbors == 0:
            no_neighbor_cells.append(cell)
            continue

        frac = counts / float(total_neighbors)
        neighbor_frac.loc[cell, frac.index] = frac.values

# Store cells with no effective neighbors
no_neighbor_cells = sorted(list(set(no_neighbor_cells)))
adata.uns['no_spatial_neighbors_k10'] = no_neighbor_cells

# Store neighbor fractions as a DataFrame in .uns (for explicit labeling)
adata.uns['neighbor_population_fractions_k10'] = neighbor_frac.copy()
adata.uns['neighbor_population_fractions_k10_df_index'] = neighbor_frac.index.to_list()
adata.uns['neighbor_population_fractions_k10_cols'] = all_pops

# Also store as a dense NumPy array in .obsm aligned with adata.obs.index
neighbor_frac_full = neighbor_frac.reindex(adata.obs.index).fillna(0.0)
adata.obsm['neighbor_population_fractions_k10'] = neighbor_frac_full.to_numpy()

# QC summary: per-population self-neighbor fraction among cells with neighbors
qc_rows = []
valid_neighbor_mask = ~neighbor_frac.index.isin(no_neighbor_cells)
for pop in all_pops:
    pop_cells = neighbor_frac.index[(obs_sub.loc[neighbor_frac.index, 'Populations'] == pop) & valid_neighbor_mask]
    if len(pop_cells) == 0:
        continue
    vals = neighbor_frac.loc[pop_cells, pop]
    qc_rows.append({
        'Population': pop,
        'n_cells_with_neighbors': int(len(pop_cells)),
        'mean_self_neighbor_fraction': float(vals.mean()),
        'median_self_neighbor_fraction': float(vals.median())
    })

qc_df = pd.DataFrame(qc_rows).sort_values('n_cells_with_neighbors', ascending=False)

n_no_neighbors = len(no_neighbor_cells)
frac_no_neighbors = n_no_neighbors / neighbor_frac.shape[0] if neighbor_frac.shape[0] > 0 else 0.0

print('Per-population mean and median fraction of same-Populations neighbors (k=10):')
print(qc_df.to_string(index=False))
print(f"\nNumber of cells with no effective spatial neighbors at k=10: {n_no_neighbors} ({frac_no_neighbors:.3%} of valid cells)")


Per-population mean and median fraction of same-Populations neighbors (k=10):
Population  n_cells_with_neighbors  mean_self_neighbor_fraction  median_self_neighbor_fraction
        PA                   30380                     0.492512                            0.5
        PB                   19947                     0.793172                            0.8
        PC                   17584                     0.186613                            0.2
        PD                   16624                     0.149723                            0.1
        PE                   16511                     0.540603                            0.6
        PF                   12248                     0.171816                            0.1
        PG                   11596                     0.735254                            0.8
        PH                   10887                     0.347506                            0.3
        PI                   10441                     0.714376    

### Agent Interpretation

The neighbor-fraction construction looks solid and is giving you a very informative QC snapshot; it sets you up well for the next aggregation and testing steps.

Key takeaways from the current step

1. **Spatial neighborhoods are well-defined and complete**
   - `no_spatial_neighbors_k10` is empty: every valid cell has at least one non-self neighbor within its own Sample_ID. This means you won’t lose power downstream due to missing neighbor features, and you can safely treat zeros in the neighbor-fraction matrix as “no neighbors of that Population,” not missing data.

2. **Strong but heterogeneous self-association across Populations**
   - Many Populations show high mean self-neighbor fractions:
     - Very high: PB (0.79), PG (0.74), PI (0.71), PQ (0.68), PR (0.73), PV (0.63), PW (0.74).
     - Moderate: PE (0.54), PK (0.55), PN (0.55), PO (0.51), PX (0.55).
   - Others are more intermingled with other Populations:
     - Low self-neighboring: PC (0.19), PD (0.15), PF (0.17), PM (0.23), PS (0.28), PY (0.26), PAA (0.18).
     - Extremely mixed: PP (0.08) and especially PZ (0.03).
   - This already hints at **distinct spatial roles**: some Populations form relatively pure domains/clusters, while others are interface / boundary / rare infiltrating types embedded in heterogeneous neighborhoods.

3. **Well-powered Populations for neighborhood-composition analyses**
   - You have large numbers of cells with neighbors in many Populations (tens of thousands for PA–PD, mid-thousands for many others). This is very favorable for your plan to define “well-powered” focal Populations.
   - Even the lower-abundance groups in this QC table (e.g., PAA with 1,027 cells) are still likely to be analyzable, provided Complexity_tertile and Purity_tertile are not extremely imbalanced across samples.

How this informs your hypothesis and next steps

Hypothesis: “For well-powered cardiac populations, the local spatial neighborhood composition shifts systematically with Complexity and Purity, revealing microenvironmental remodeling not explained by intrinsic maturation alone.”

The current step doesn’t yet condition on Complexity or Purity, but it does tell you:

- **Which Populations are promising focal Populations** for testing Complexity/Purity-dependent neighborhood shifts.
- **Baseline neighborhood structure** against which shifts can be interpreted.

Specific guidance for next steps:

1. **Choice of focal Populations**
   - Prioritize Populations that:
     - Have high `n_cells_with_neighbors`.
     - Show **intermediate** self-neighbor fractions (not ~0 and not ~1), because they have room for meaningful shifts in neighbor composition.
   - Particularly promising focal Populations:
     - PA (n=30k; mean self=0.49)
     - PE (16.5k; 0.54), PH (10.9k; 0.35), PJ (9.5k; 0.38), PK (8.5k; 0.55), PL (8.1k; 0.41), PM (7.4k; 0.23), PN (7.3k; 0.55), PO (5.8k; 0.51), PT (3.7k; 0.44), PU (2.4k; 0.31), PX (1.6k; 0.55), PY (1.3k; 0.26).
   - Highly self-clustered types (PB, PG, PI, PQ, PR, PV, PW) are still interesting: a modest *relative* change in neighbor fractions could be biologically meaningful, but the dynamic range is somewhat constrained (their neighbors are mostly conspecifics).

2. **Populations especially suited for revealing microenvironmental remodeling**
   - Very mixed/self-low Populations (PC, PD, PF, PM, PS, PY, PP, PZ, PAA) are likely “interface” or “niche” populations. Changes in Complexity or Purity may manifest as:
     - Shifts in which other Populations form their microenvironment.
     - Redistribution between being adjacent to more immature vs more mature neighbors (once you have Complexity/Purity stratified).
   - PZ (mean self 0.03) and PP (0.08) are particularly striking: they almost never see same-type neighbors. For these, **composition of other Populations** around them (e.g., %PA, %PG, etc.) is likely to be highly informative about their microenvironment and how it remodels.

3. **Design of the aggregation step**
   - In the next step, when aggregating `neighbor_population_fractions_k10` by focal Population × Complexity_tertile × Purity_tertile × Sample_ID:
     - Enforce a **minimum per-stratum cell count**, e.g.:
       - ≥ 30 cells per (focal Pop × Complexity_tertile × Sample_ID) and similarly for Purity, if feasible.
     - Require a minimum number of Sample_IDs with both low and high tertiles represented for each focal Population (e.g., at least 3–5 samples) to support your “not driven by single Sample_ID” criterion.
   - You may want to compute and store:
     - For each focal Population and neighbor Population: mean, median, variance, and maybe interquartile range of neighbor fractions per tertile and per sample.
     - The **proportion of focal cells that ever see ≥1 neighbor** of a given neighbor Population (a binary “has at least one neighbor of type X”), as a complementary robustness check to fraction-based analyses.

4. **Pre-visualization before formal tests**
   - To avoid rediscovering what the original paper did, lean into **multi-dimensional summaries** rather than just pairwise co-localization:
     - For a few focal Populations (e.g., PA, PM, PZ), plot neighbor-fraction profiles (across all neighbor Populations) as **radar or bar plots** stratified by low vs high Complexity_tertile, and separately for Purity.
     - Make per-sample effect plots: within each focal Population, show Δ mean neighbor fraction (high–low) per neighbor Population per Sample_ID, to assess consistency in sign and magnitude before pooled testing.
   - This will help you identify which focal–neighbor pairs exhibit the clearest and most consistent remodeling patterns to prioritize in downstream narrative.

5. **Consider scale and k choice as sensitivity checks**
   - You’ve used k=10 for neighbor definition. For the main analysis, this is reasonable and you’ve confirmed there are no edge issues.
   - Later (optionally, not to derail the current plan), you might:
     - Repeat the pipeline for another k (e.g., 5 or 20) for a subset of focal Populations to confirm that detected Complexity/Purity effects are not hyper-sensitive to k.
   - If some Populations (e.g., extremely dense clusters such as PB, PG, PI) are overwhelmingly self-neighboring, you might examine **non-self neighbor fractions** (renormalizing after excluding self-type neighbors) as a sensitivity analysis to see whether the mix of “other” populations changes with Complexity/Purity.

6. **Anticipated patterns relevant to the hypothesis**
   - You’ll want to look for focal Populations where:
     - As Complexity increases, the fraction of neighbors that are more “mature-like” Populations increases (or decreases) in a consistent direction across samples, even after controlling for focal cell identity (since you are within a focal Pop).
     - As Purity increases (presumably indicating clearer transcriptional identity), the neighborhood becomes more segregated or, conversely, more mixed with specific partner Populations.
   - Particularly for very mixed Populations (PC, PD, PF, PM, PS, PY, PZ, PAA), robust, directionally concordant shifts in neighbor composition with Complexity/Purity would **strongly support** the idea of microenvironmental remodeling beyond intrinsic maturation.

7. **Implementation checks before proceeding**
   - Confirm that `neighbor_population_fractions_k10` in `.obsm` aligns exactly with `adata.obs.index` (the code should ensure this, but a one-line assertion can be helpful):
     - `assert adata.obsm['neighbor_population_fractions_k10'].shape == (adata.n_obs, len(all_pops))`
   - Make sure Complexity_tertile and Purity_tertile are defined only for `valid_cells`. For cells outside `valid_cells`, their neighbor fractions exist but should be excluded from analyses if their metadata is incomplete.

How this relates to hypothesis validation so far

- At this stage you’ve **established a robust, per-cell spatial neighborhood feature set** and confirmed distinct baseline spatial clustering vs mixing across Populations.
- This is necessary but not sufficient to validate the hypothesis; you still need:
  - Stratification by Complexity_tertile and Purity_tertile.
  - Per-sample-stratified Wilcoxon testing of neighbor fractions within focal Populations.
  - Cross-sample consistency checks and variance attribution.

The current results are promising in that they show:
- Adequate power across many Populations.
- Clear differences in baseline neighborhood structure, which gives room to detect systematic remodeling with Complexity and Purity in the upcoming steps.

## Next Steps
Step 1: Aggregate per-cell neighbor-fraction features from `adata.obsm['neighbor_population_fractions_k10']` by focal Population × Complexity_tertile × Purity_tertile × Sample_ID, computing per-neighbor-Population summary statistics (mean, median, variance, and cell counts), define well-powered focal Populations separately for Complexity and Purity axes based on minimum per-stratum counts and balanced low vs high tertile representation across samples, and record which focal–neighbor strata are themselves well powered for downstream testing.
Step 2: Within each well-powered focal Population, run per-neighbor-Population Wilcoxon rank-sum tests on per-cell neighbor fractions to compare low vs high Complexity_tertile and, separately, low vs high Purity_tertile (optionally restricting primary Complexity tests to cells within a matched Purity tertile as a sensitivity analysis), applying Benjamini–Hochberg FDR correction within each focal Population × axis, and compute per-sample effect directions to flag focal–neighbor pairs whose significance is driven by a single Sample_ID or lack of cross-sample concordance.
Step 3: Summarize, for each focal Population, the neighbor Populations showing the most robust and largest Complexity- and Purity-associated shifts in neighborhood composition, emphasizing effects with consistent per-sample directions and noting whether they occur in highly mixed versus highly self-clustered focal Populations to interpret these as microenvironmental remodeling not explained by intrinsic maturation alone.

## This code aggregates per-cell neighbor-fraction features by focal Population × Complexity/Purity tertile × Sample_ID, computes summary statistics and power filters per neighbor Population stratum, and defines well-powered focal Populations separately for Complexity and Purity axes, storing all results in adata.uns for use in the downstream Wilcoxon testing step.

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Aggregate per-cell neighborhood compositions and define well-powered focal Populations and strata

# Shorthand
adata_local = adata

# Basic checks
if 'neighbor_population_fractions_k10' not in adata_local.obsm_keys():
    raise ValueError("neighbor_population_fractions_k10 not found in adata.obsm; run the spatial KNN neighborhood step first.")

required_obs_cols = ['Populations', 'Sample_ID', 'Complexity_tertile', 'Purity_tertile']
missing_cols = [c for c in required_obs_cols if c not in adata_local.obs.columns]
if missing_cols:
    raise ValueError(f"Missing required metadata columns in adata.obs: {missing_cols}")

# Recover population labels for neighbor-fraction columns
pop_cols = adata_local.uns.get('neighbor_population_fractions_k10_cols', None)
if pop_cols is None:
    pop_cols = adata_local.obs['Populations'].astype('category').cat.categories.tolist()

neighbor_mat = adata_local.obsm['neighbor_population_fractions_k10']
if neighbor_mat.shape[1] != len(pop_cols):
    raise ValueError("Mismatch between neighbor_population_fractions_k10 columns and stored pop_cols.")

# Optional strict shape check for safety
if neighbor_mat.shape[0] != adata_local.n_obs:
    raise ValueError("neighbor_population_fractions_k10 row dimension does not match adata.n_obs; index misalignment suspected.")

# Build a DataFrame with per-cell neighbor fractions and key metadata
nf_df = pd.DataFrame(
    neighbor_mat,
    index=adata_local.obs.index,
    columns=pop_cols
)

meta_cols = ['Populations', 'Sample_ID', 'Complexity_tertile', 'Purity_tertile']
nf_df = nf_df.join(adata_local.obs[meta_cols])

# Restrict to cells with complete tertile information
complete_mask = nf_df[['Complexity_tertile', 'Purity_tertile']].notna().all(axis=1)
nf_df = nf_df.loc[complete_mask].copy()

# Helper to aggregate by a given tertile axis and mark well-powered focal populations and strata
def aggregate_by_tertile(nf_df, tertile_col, min_cells_per_stratum=30, min_samples_per_tert=2):
    """Aggregate neighbor fractions by focal Population × tertile × Sample_ID and define well-powered focal Populations and strata."""
    group_keys = ['Populations', tertile_col, 'Sample_ID']
    value_cols = pop_cols

    # Melt neighbor fractions to long format for flexible aggregation
    melted = nf_df.reset_index().rename(columns={'index': 'cell_id'}).melt(
        id_vars=['cell_id'] + group_keys,
        value_vars=value_cols,
        var_name='neighbor_Population',
        value_name='neighbor_fraction'
    )

    # Compute per-stratum (focal Pop × tertile × Sample_ID × neighbor_Population) stats
    agg = (
        melted
        .groupby(group_keys + ['neighbor_Population'])
        .agg(
            n_cells=('cell_id', 'nunique'),
            mean_fraction=('neighbor_fraction', 'mean'),
            median_fraction=('neighbor_fraction', 'median'),
            var_fraction=('neighbor_fraction', 'var')
        )
        .reset_index()
    )

    # Mark well-powered focal–neighbor strata based on minimum cell count
    agg['is_stratum_well_powered'] = agg['n_cells'] >= min_cells_per_stratum

    # Derive focal-Population–level power summaries: how many samples contribute powered low vs high tertiles (for any neighbor)
    stratum_power_any = (
        agg
        .groupby(['Populations', tertile_col, 'Sample_ID'])['is_stratum_well_powered']
        .any()
        .reset_index()
    )

    sample_counts = (
        stratum_power_any[stratum_power_any['is_stratum_well_powered']]
        .groupby(['Populations', tertile_col])['Sample_ID']
        .nunique()
        .reset_index(name='n_samples_powered')
    )

    sample_counts_pivot = sample_counts.pivot_table(
        index='Populations',
        columns=tertile_col,
        values='n_samples_powered',
        fill_value=0
    )
    sample_counts_pivot.columns = [f"{tertile_col}_{c}_n_samples_powered" for c in sample_counts_pivot.columns]

    # Also compute total cells per focal Pop × tertile across all samples
    cell_counts = (
        nf_df
        .groupby(['Populations', tertile_col])
        .size()
        .reset_index(name='n_cells_total')
    )
    cell_counts_pivot = cell_counts.pivot_table(
        index='Populations',
        columns=tertile_col,
        values='n_cells_total',
        fill_value=0
    )
    cell_counts_pivot.columns = [f"{tertile_col}_{c}_n_cells" for c in cell_counts_pivot.columns]

    # Combine into a focal-Population summary
    pop_summary = sample_counts_pivot.join(cell_counts_pivot, how='outer').fillna(0).reset_index()

    # Define well-powered focal Populations for this tertile axis:
    # require at least min_cells_per_stratum cells in both low and high tertiles and
    # at least min_samples_per_tert samples with any well-powered stratum in both low and high tertiles.
    low_cells = pop_summary.get(f"{tertile_col}_low_n_cells", pd.Series(0, index=pop_summary.index))
    high_cells = pop_summary.get(f"{tertile_col}_high_n_cells", pd.Series(0, index=pop_summary.index))
    low_samps = pop_summary.get(f"{tertile_col}_low_n_samples_powered", pd.Series(0, index=pop_summary.index))
    high_samps = pop_summary.get(f"{tertile_col}_high_n_samples_powered", pd.Series(0, index=pop_summary.index))

    pop_summary[f'well_powered_{tertile_col}'] = (
        (low_cells >= min_cells_per_stratum) &
        (high_cells >= min_cells_per_stratum) &
        (low_samps >= min_samples_per_tert) &
        (high_samps >= min_samples_per_tert)
    )

    return agg, pop_summary

# Aggregate for Complexity tertiles
complexity_agg, complexity_pop_summary = aggregate_by_tertile(
    nf_df,
    tertile_col='Complexity_tertile',
    min_cells_per_stratum=30,
    min_samples_per_tert=2
)

# Aggregate for Purity tertiles
purity_agg, purity_pop_summary = aggregate_by_tertile(
    nf_df,
    tertile_col='Purity_tertile',
    min_cells_per_stratum=30,
    min_samples_per_tert=2
)

# Identify focal Populations that are well powered for Complexity and/or Purity axes
complexity_well = set(complexity_pop_summary.loc[complexity_pop_summary['well_powered_Complexity_tertile'], 'Populations'])
purity_well = set(purity_pop_summary.loc[purity_pop_summary['well_powered_Purity_tertile'], 'Populations'])

focal_pops_well_powered = sorted(list(complexity_well.union(purity_well)))

print("\n=== Focal Populations well powered for Complexity and/or Purity tertile comparisons ===")
print(f"Total well-powered focal Populations (Complexity or Purity): {len(focal_pops_well_powered)}")
print(", ".join(focal_pops_well_powered))

print("\nComplexity-based focal Population summary (top 20 by total cells):")
complexity_pop_summary['total_cells'] = complexity_pop_summary[[c for c in complexity_pop_summary.columns if c.endswith('_n_cells')]].sum(axis=1)
print(complexity_pop_summary.sort_values('total_cells', ascending=False).head(20).to_string(index=False))

print("\nPurity-based focal Population summary (top 20 by total cells):")
purity_pop_summary['total_cells'] = purity_pop_summary[[c for c in purity_pop_summary.columns if c.endswith('_n_cells')]].sum(axis=1)
print(purity_pop_summary.sort_values('total_cells', ascending=False).head(20).to_string(index=False))

# Store detailed aggregation results and power summaries in adata.uns for downstream testing
adata_local.uns['neighbor_fraction_agg_by_Complexity_tertile'] = complexity_agg
adata_local.uns['neighbor_fraction_agg_by_Purity_tertile'] = purity_agg
adata_local.uns['neighbor_fraction_pop_summary_Complexity'] = complexity_pop_summary
adata_local.uns['neighbor_fraction_pop_summary_Purity'] = purity_pop_summary
adata_local.uns['neighbor_fraction_focal_pops_well_powered'] = focal_pops_well_powered



=== Focal Populations well powered for Complexity and/or Purity tertile comparisons ===
Total well-powered focal Populations (Complexity or Purity): 26
PA, PAA, PB, PC, PD, PE, PF, PG, PH, PI, PJ, PK, PL, PM, PN, PO, PP, PQ, PR, PT, PU, PV, PW, PX, PY, PZ

Complexity-based focal Population summary (top 20 by total cells):
Populations  Complexity_tertile_low_n_samples_powered  Complexity_tertile_mid_n_samples_powered  Complexity_tertile_high_n_samples_powered  Complexity_tertile_low_n_cells  Complexity_tertile_mid_n_cells  Complexity_tertile_high_n_cells  well_powered_Complexity_tertile  total_cells
         PA                                       3.0                                       3.0                                        3.0                         10474.0                         12193.0                           7713.0                             True      30380.0
         PB                                       3.0                                       3.0                

### Agent Interpretation

The aggregation / power-screening step looks solid and sets you up well for the hypothesis you’re testing. A few key points and suggested refinements for the next steps:

1. **You now have a strong set of focal Populations for neighborhood analysis**

- You have 26 focal Populations (PA–PZ except PS) that are well powered on at least one axis. Many have:
  - Thousands of cells per tertile, and
  - ≥3 powered samples per low and high tertile for both Complexity and Purity.
- For the hypothesis (systematic shifts “consistent across samples rather than driven by single-sample idiosyncrasies”), this cross-sample coverage is exactly what you need.

2. **Complexity vs Purity balance looks good but not symmetric for all Populations**

- Some Populations are “front-loaded” in low vs high tertiles on one axis. For example:
  - PB (Complexity): heavy low-tertile count (17k) vs high (717), but still passes your minimal thresholds.
  - PB (Purity): the inverse—high Purity tertile is dominant (17k).
- This suggests:
  - You’ll have more power to detect neighbor shifts driving transitions where the bulk of cells lie (PB: low→mid Complexity and mid→high Purity).
  - For extreme tertiles with relatively few cells per Population, effect sizes will need to be large to be robust.

In downstream testing, I’d:
- Still include PB, but be cautious interpreting small absolute differences in the very small strata (e.g. PB high Complexity).
- For the final biological summary, prioritize Populations where *both* low and high tertiles have reasonably large counts (e.g. PA, PC–PE, PF–PH, PN, PM), because their effects will be less sensitive to distributional quirks.

3. **Stratum-level power is defined; use it to filter neighbor pairs in the next step**

You already mark `is_stratum_well_powered` per:
- focal Pop × (Complexity or Purity tertile) × Sample × neighbor_Pop

For the Wilcoxon tests in the next step:

- Before testing low vs high tertiles for a given focal–neighbor pair, require:
  - In **each tertile**, at least:
    - `min_cells_per_stratum` cells aggregated across samples, *and*
    - ≥`min_samples_per_tert` samples contributing **powered** strata.
- This will:
  - Avoid unstable comparisons for rare neighbor Populations.
  - Focus on neighbor changes that are reasonably well-sampled across samples, in line with your cross-sample consistency criterion.

4. **Plan for cross-sample consistency checks**

The current step gives you per-cell data plus a per-stratum summary; the next step should explicitly encode per-sample effect directions. A clear structure:

- For each focal Population F and neighbor Population N:
  - For each sample S with enough powered cells in both low and high tertiles:
    - Compute the **per-sample effect** (e.g., mean neighbor fraction high − low).
    - Record its sign and magnitude.
  - Run a **global Wilcoxon test** on all per-cell fractions (or per-cell within matched Purity tertile if you do that sensitivity analysis).
  - Filter significant F–N pairs to those where:
    - A high fraction of samples (e.g. ≥2/3) show the same sign of effect.
    - There are at least 2–3 such concordant samples.

This will let you identify changes that are not driven by a single sample, as mandated by the hypothesis.

5. **Distinguish Complexity vs Purity effects carefully**

Because the dataset demonstrates substantial shifts in total cell counts across tertiles (e.g. PB: Complexity skews low, Purity skews high), there’s a real risk of confounding:

- For **Complexity** effects:
  - It’s a good idea to implement the proposed sensitivity analysis: restrict comparisons to cells within the same Purity tertile (e.g. low-Complexity vs high-Complexity *within Purity-mid*).
  - Start with the tertile that has abundant cells for that focal Population (often mid-Purity).
- For **Purity** effects:
  - Similarly, consider matching on Complexity tertile (e.g. within Complexity-mid). Even if you don’t do this everywhere, running it on a few focal Populations (e.g. PA, PC, PE, PM) will help argue that neighborhood shifts with Purity are not merely reflecting Complexity shifts.

You can then classify F–N pairs into:
- “Robust to matching” (effect holds in the matched-stratum sensitivity analysis).
- “Attenuated when matching” (could be partly mediated by Complexity or Purity rather than independent microenvironmental remodeling).

6. **Interpreting “microenvironmental remodeling” vs “self-clustering”**

You already have the ingredients in the neighbor-fraction matrix to determine whether a focal Population is:
- **Self-clustered**: high median self-neighbor fraction (e.g. PA cells mostly surrounded by PA).
- **Highly mixed**: broader distribution of neighbor types.

For each focal Population:
- Quantify baseline **self-neighborhood**:
  - E.g. median fraction of neighbors that are the same Population (PA around PA).
- Identify which neighbor Fractions drive tertile differences:
  - Are shifts primarily in self-fraction (e.g. PA surrounded more by PA at higher Complexity)?
  - Or in heterotypic neighbors (e.g. PA gains PB neighbors and loses PD neighbors as Purity increases)?

The hypothesis is specifically about microenvironmental remodeling beyond intrinsic maturation, so in interpretation:

- Changes in **self-neighbor fraction only** could be:
  - Partly driven by internal clustering/maturation of that Population.
- Changes involving **distinct neighbor Populations** (e.g. PA gaining PF and losing PG neighbors with higher Complexity, in a consistent fashion across samples) are more compelling evidence of systematic microenvironmental reorganization.

7. **Prioritizing focal Populations for deep follow-up**

Based on the summaries:

- **Highest power and best balance**:
  - PA, PC, PD, PE, PF, PH, PN, PQ, PT (large total cells, good distribution across tertiles on both axes).
- **Interesting asymmetries worth examining**:
  - PB, PG, PI, PJ, PM:
    - These will likely show strong Purity or Complexity effects but could be biased toward one extremal tertile. 
    - For them, cross-sample directionality will be crucial.

For these focal Populations, plan to:

- Produce summary visuals after the testing step:
  - Heatmaps: neighbor Populations (columns) × tertiles (rows), colored by mean fraction, faceted by focal Pop.
  - Effect-size barplots: for each focal Pop, show Δfraction (high − low) per neighbor with FDR-significant and cross-sample-consistent effects, separately for Complexity and Purity.
- These visual summaries will directly support or refute the idea of “systematic, consistent remodeling” per Population.

8. **Code-wise, minor thoughts for robustness**

Nothing critical, but for later steps:

- Consider storing the **per-sample, per- tertile, per-focal–neighbor summary** table in `adata.uns` as well (not just the grand-mean per axis), so the next-step testing code can directly reuse it for cross-sample consistency instead of recomputing from per-cell.
- When you later recombine Complexity and Purity results, ensure the same naming convention for axes and p-values (e.g., `pval_complexity`, `fdr_complexity`, `pval_purity`, `fdr_purity`) to keep the interpretation clean.

Overall, this step confirms you have a sufficiently broad and well-powered set of cardiac Populations (PA–PZ minus PS) for robust neighborhood-composition testing along both Complexity and Purity. The next step should lean heavily on (i) per-focal–neighbor Wilcoxon tests with robust power filtering and (ii) per-sample directionality checks, then (iii) interpretation framed around self vs non-self neighbor shifts to decide which changes best support microenvironmental remodeling beyond intrinsic maturation.

## Next Steps
Step 1: Within each well-powered focal Population and for each neighbor Population, perform Wilcoxon rank-sum tests on per-cell neighbor fractions to compare low vs high Complexity_tertile and, separately, low vs high Purity_tertile, enforcing minimum per-tertile cell and sample counts per Sample_ID for inclusion in both pooled tests and per-sample effect summaries, applying Benjamini–Hochberg FDR correction per axis across all focal–neighbor pairs, and annotating each result with per-sample effect directions, concordance metrics, and self- vs heterotypic neighbor flags.
Step 2: Summarize, for each focal Population, the neighbor Populations showing the strongest and most cross-sample-consistent Complexity- and Purity-associated shifts in neighborhood composition, explicitly distinguishing changes dominated by self-neighbor fractions from those driven by heterotypic neighbors, and generate text tables that report effect sizes, FDR values, and cross-sample concordance as evidence for or against microenvironmental remodeling beyond intrinsic maturation.

## This code tests, for each focal cell population, whether the fraction of different neighboring populations changes between low vs high Complexity or Purity tertiles using Wilcoxon rank-sum tests, while enforcing per-sample power filters. It then applies Benjamini–Hochberg FDR correction, summarizes per-sample effect directions and self vs heterotypic interactions, and prints significant, cross-sample-concordant neighbor shifts for biological interpretation of how microenvironments vary with tumor Complexity and Purity.

In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import ranksums

# Step: Per-focal-population Wilcoxon tests of neighbor fractions vs Complexity and Purity tertiles,
# with per-sample effect directions, self/heterotypic flags, and BH-FDR correction per axis.

# -----------------------------------------------------------------------------
# Helper: Benjamini–Hochberg FDR
# -----------------------------------------------------------------------------
def benjamini_hochberg(pvals):
    """Return BH-adjusted p-values as a NumPy array (same shape as pvals)."""
    pvals = np.asarray(pvals, dtype=float)
    n = pvals.size
    order = np.argsort(pvals)
    ranks = np.arange(1, n + 1)
    adj = np.empty(n, dtype=float)
    adj[order] = pvals[order] * n / ranks
    # Enforce monotonicity
    for i in range(n - 2, -1, -1):
        adj[order[i]] = min(adj[order[i]], adj[order[i + 1]])
    return np.minimum(adj, 1.0)

# -----------------------------------------------------------------------------
# Retrieve per-cell neighbor fractions and metadata
# -----------------------------------------------------------------------------
if 'neighbor_population_fractions_k10' not in adata.obsm_keys():
    raise ValueError("neighbor_population_fractions_k10 not found in adata.obsm; run neighborhood construction first.")

neighbor_mat = adata.obsm['neighbor_population_fractions_k10']
pop_cols = adata.uns.get('neighbor_population_fractions_k10_cols', None)
if pop_cols is None:
    pop_cols = adata.obs['Populations'].astype('category').cat.categories.tolist()

if neighbor_mat.shape[1] != len(pop_cols):
    raise ValueError("Mismatch between neighbor_population_fractions_k10 columns and pop_cols.")

# Build DataFrame with neighbor fractions and key obs metadata
nf_df = pd.DataFrame(neighbor_mat, index=adata.obs.index, columns=pop_cols)
meta_cols = ['Populations', 'Sample_ID', 'Complexity_tertile', 'Purity_tertile']
missing_meta = [c for c in meta_cols if c not in adata.obs.columns]
if missing_meta:
    raise ValueError(f"Missing required metadata columns in adata.obs: {missing_meta}")

nf_df = nf_df.join(adata.obs[meta_cols])

# Restrict to cells with complete tertile info
mask_complete = nf_df[['Complexity_tertile', 'Purity_tertile']].notna().all(axis=1)
nf_df = nf_df.loc[mask_complete].copy()

# Focal Populations pre-screened as well powered (Complexity and/or Purity)
well_focal = adata.uns.get('neighbor_fraction_focal_pops_well_powered', None)
if well_focal is None:
    # Fallback: all Populations present
    well_focal = nf_df['Populations'].astype('category').cat.categories.tolist()
else:
    well_focal = list(well_focal)

pop_cols = list(pop_cols)  # ensure list

# -----------------------------------------------------------------------------
# Parameters for power filtering
# -----------------------------------------------------------------------------
min_cells_per_tertile_global = 30        # per focal × axis group (all samples pooled)
min_samples_per_tertile_global = 2       # per focal × axis group (number of samples with any cells)
min_cells_per_sample_tertile = 5        # per-sample minimum per tertile to include that sample in pooled test and per-sample effects

# -----------------------------------------------------------------------------
# Core testing function: one axis (Complexity_tertile or Purity_tertile)
# -----------------------------------------------------------------------------

def run_axis_tests(axis_col):
    """Run Wilcoxon tests for low vs high tertile of axis_col for all focal × neighbor pairs.

    Returns a DataFrame of results with columns:
    focal_Population, neighbor_Population, axis, n_cells_low, n_cells_high,
    n_samples_low, n_samples_high, mean_frac_low, mean_frac_high, delta_mean,
    pval, fdr, n_samples_effect, n_samples_concordant, frac_concordant,
    is_self_neighbor, cross_sample_concordant.
    """
    results = []

    # Iterate over focal Populations
    for focal in well_focal:
        focal_cells_all = nf_df[nf_df['Populations'] == focal]
        if focal_cells_all.empty:
            continue

        # Define low/high groups for this axis
        low_mask_all = focal_cells_all[axis_col] == 'low'
        high_mask_all = focal_cells_all[axis_col] == 'high'
        if low_mask_all.sum() < min_cells_per_tertile_global or high_mask_all.sum() < min_cells_per_tertile_global:
            continue

        # Sample-level coverage (any cells)
        low_samples_any = focal_cells_all.loc[low_mask_all, 'Sample_ID'].nunique()
        high_samples_any = focal_cells_all.loc[high_mask_all, 'Sample_ID'].nunique()
        if (low_samples_any < min_samples_per_tertile_global) or (high_samples_any < min_samples_per_tertile_global):
            continue

        # Build per-sample filtered masks enforcing per-sample minima
        include_cells_mask = pd.Series(False, index=focal_cells_all.index)
        per_sample_info = {}
        for sid, sub in focal_cells_all.groupby('Sample_ID'):
            sub_low = (sub[axis_col] == 'low')
            sub_high = (sub[axis_col] == 'high')
            if sub_low.sum() >= min_cells_per_sample_tertile and sub_high.sum() >= min_cells_per_sample_tertile:
                include_cells_mask.loc[sub.index] = True
                per_sample_info[sid] = {
                    'low_mask': sub_low,
                    'high_mask': sub_high
                }

        # Restrict to per-sample-powered cells
        focal_cells = focal_cells_all.loc[include_cells_mask]
        if focal_cells.empty:
            continue

        # Recompute low/high masks after per-sample filtering
        low_mask = focal_cells[axis_col] == 'low'
        high_mask = focal_cells[axis_col] == 'high'
        if low_mask.sum() < min_cells_per_tertile_global or high_mask.sum() < min_cells_per_tertile_global:
            continue

        low_samples = focal_cells.loc[low_mask, 'Sample_ID'].nunique()
        high_samples = focal_cells.loc[high_mask, 'Sample_ID'].nunique()
        if (low_samples < min_samples_per_tertile_global) or (high_samples < min_samples_per_tertile_global):
            continue

        # For each neighbor Population, test low vs high
        for neigh in pop_cols:
            x_low = focal_cells.loc[low_mask, neigh].values
            x_high = focal_cells.loc[high_mask, neigh].values

            # Require at least some non-zero or variable data in each group
            if (x_low.size < min_cells_per_tertile_global) or (x_high.size < min_cells_per_tertile_global):
                continue
            if (np.allclose(x_low, x_low.mean()) and np.allclose(x_high, x_high.mean()) and
                np.isclose(x_low.mean(), x_high.mean())):
                # Completely flat, identical distributions
                continue

            # Global Wilcoxon rank-sum on per-cell fractions (high vs low)
            try:
                stat, pval = ranksums(x_high, x_low)
            except ValueError:
                continue

            mean_low = float(np.mean(x_low))
            mean_high = float(np.mean(x_high))
            delta = mean_high - mean_low

            # Per-sample effect directions (using only samples that passed per-sample minima)
            per_sample_deltas = []
            for sid, sub in focal_cells.groupby('Sample_ID'):
                sub_low = sub.loc[sub[axis_col] == 'low', neigh].values
                sub_high = sub.loc[sub[axis_col] == 'high', neigh].values
                if (sub_low.size >= min_cells_per_sample_tertile) and (sub_high.size >= min_cells_per_sample_tertile):
                    d = float(np.mean(sub_high) - np.mean(sub_low))
                    per_sample_deltas.append(d)

            if len(per_sample_deltas) == 0:
                n_samp_eff = 0
                n_conc = 0
                frac_conc = np.nan
            else:
                n_samp_eff = len(per_sample_deltas)
                signs = np.sign(per_sample_deltas)
                signs = signs[signs != 0]
                if signs.size == 0:
                    n_conc = 0
                    frac_conc = 0.0
                else:
                    pos = np.sum(signs > 0)
                    neg = np.sum(signs < 0)
                    n_conc = max(pos, neg)
                    frac_conc = n_conc / float(signs.size)

            is_self = (focal == neigh)

            results.append({
                'focal_Population': focal,
                'neighbor_Population': neigh,
                'axis': axis_col,
                'n_cells_low': int(x_low.size),
                'n_cells_high': int(x_high.size),
                'n_samples_low': int(low_samples),
                'n_samples_high': int(high_samples),
                'mean_frac_low': mean_low,
                'mean_frac_high': mean_high,
                'delta_mean_high_minus_low': delta,
                'pval': float(pval),
                'n_samples_effect': int(n_samp_eff),
                'n_samples_concordant': int(n_conc),
                'frac_concordant': float(frac_conc) if not np.isnan(frac_conc) else np.nan,
                'is_self_neighbor': bool(is_self)
            })

    if not results:
        return pd.DataFrame(columns=[
            'focal_Population', 'neighbor_Population', 'axis', 'n_cells_low', 'n_cells_high',
            'n_samples_low', 'n_samples_high', 'mean_frac_low', 'mean_frac_high',
            'delta_mean_high_minus_low', 'pval', 'fdr', 'n_samples_effect',
            'n_samples_concordant', 'frac_concordant', 'is_self_neighbor',
            'cross_sample_concordant'
        ])

    res_df = pd.DataFrame(results)

    # BH-FDR correction within this axis across all focal–neighbor pairs
    fdr = benjamini_hochberg(res_df['pval'].values)
    res_df['fdr'] = fdr

    # Define a boolean cross-sample concordance flag using reasonable thresholds
    frac_conc_thresh = 0.6
    min_samples_effect = 2
    res_df['cross_sample_concordant'] = (
        (res_df['n_samples_effect'] >= min_samples_effect) &
        (res_df['frac_concordant'] >= frac_conc_thresh)
    )

    return res_df

# -----------------------------------------------------------------------------
# Run tests for Complexity_tertile and Purity_tertile
# -----------------------------------------------------------------------------
complexity_results = run_axis_tests('Complexity_tertile')
purity_results = run_axis_tests('Purity_tertile')

adata.uns['neighbor_fraction_wilcoxon_Complexity'] = complexity_results
adata.uns['neighbor_fraction_wilcoxon_Purity'] = purity_results

# -----------------------------------------------------------------------------
# Text summaries: per-focal-population top hits, split by self vs heterotypic
# -----------------------------------------------------------------------------

fdr_thresh = 0.05

print("\n=== Complexity_tertile: Significant neighbor shifts per focal Population (FDR <= 0.05) ===")
if complexity_results.shape[0] == 0:
    print("No tests were run or no results available.")
else:
    comp_sig = complexity_results[complexity_results['fdr'] <= fdr_thresh].copy()
    if comp_sig.empty:
        print("No focal–neighbor pairs passed FDR threshold.")
    else:
        for focal in sorted(comp_sig['focal_Population'].unique()):
            sub = comp_sig[comp_sig['focal_Population'] == focal].copy()
            sub = sub.sort_values(by=['cross_sample_concordant', 'fdr', 'delta_mean_high_minus_low'],
                                  ascending=[False, True, False])
            print(f"\n--- Focal Population: {focal} ---")
            if sub.empty:
                print("No significant neighbor shifts.")
                continue
            # Self-neighbor effects
            self_sub = sub[sub['is_self_neighbor']]
            hetero_sub = sub[~sub['is_self_neighbor']]
            if not self_sub.empty:
                print("Self-neighbor shifts:")
                print(self_sub.head(10)[[
                    'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                    'n_samples_effect', 'n_samples_concordant', 'frac_concordant',
                    'cross_sample_concordant'
                ]].to_string(index=False))
            if not hetero_sub.empty:
                print("Heterotypic neighbor shifts:")
                print(hetero_sub.head(10)[[
                    'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                    'n_samples_effect', 'n_samples_concordant', 'frac_concordant',
                    'cross_sample_concordant'
                ]].to_string(index=False))

print("\n=== Purity_tertile: Significant neighbor shifts per focal Population (FDR <= 0.05) ===")
if purity_results.shape[0] == 0:
    print("No tests were run or no results available.")
else:
    pur_sig = purity_results[purity_results['fdr'] <= fdr_thresh].copy()
    if pur_sig.empty:
        print("No focal–neighbor pairs passed FDR threshold.")
    else:
        for focal in sorted(pur_sig['focal_Population'].unique()):
            sub = pur_sig[pur_sig['focal_Population'] == focal].copy()
            sub = sub.sort_values(by=['cross_sample_concordant', 'fdr', 'delta_mean_high_minus_low'],
                                  ascending=[False, True, False])
            print(f"\n--- Focal Population: {focal} ---")
            if sub.empty:
                print("No significant neighbor shifts.")
                continue
            self_sub = sub[sub['is_self_neighbor']]
            hetero_sub = sub[~sub['is_self_neighbor']]
            if not self_sub.empty:
                print("Self-neighbor shifts:")
                print(self_sub.head(10)[[
                    'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                    'n_samples_effect', 'n_samples_concordant', 'frac_concordant',
                    'cross_sample_concordant'
                ]].to_string(index=False))
            if not hetero_sub.empty:
                print("Heterotypic neighbor shifts:")
                print(hetero_sub.head(10)[[
                    'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                    'n_samples_effect', 'n_samples_concordant',
                    'frac_concordant', 'cross_sample_concordant'
                ]].to_string(index=False))



=== Complexity_tertile: Significant neighbor shifts per focal Population (FDR <= 0.05) ===

--- Focal Population: PA ---
Self-neighbor shifts:
neighbor_Population  delta_mean_high_minus_low          fdr  n_samples_effect  n_samples_concordant  frac_concordant  cross_sample_concordant
                 PA                  -0.063096 1.848850e-84                 3                     3              1.0                     True
Heterotypic neighbor shifts:
neighbor_Population  delta_mean_high_minus_low          fdr  n_samples_effect  n_samples_concordant  frac_concordant  cross_sample_concordant
                 PN                   0.037355 1.669891e-77                 3                     3         1.000000                     True
                 PE                   0.023600 4.165501e-33                 3                     3         1.000000                     True
                 PC                  -0.019286 6.443576e-20                 3                     3         1.000000 

### Agent Interpretation

These results are strongly supportive of the hypothesis and give you a clear set of “microenvironmental remodeling modules” to carry forward.

Key points and how they bear on the hypothesis:

1. **Widespread, robust shifts in neighborhood composition**

   - For essentially every well-powered focal population (PA, PB, PC, PD, PE, PF, PG, PH, PI, PJ, PK, PL, PM, PN, PO, PP, PQ, PR, PT, PU, PV, PW, PX, PY, PZ), you see:
     - Highly significant Wilcoxon tests (FDR often << 1e‑10).
     - Substantial mean shifts in neighbor fractions (|delta_mean| up to ~0.3–0.4).
     - High cross-sample concordance (frac_concordant ≥ 0.66, usually 1.0 across all 3 powered samples).
   - This is exactly the pattern you would expect if spatial neighborhoods are being systematically reconfigured rather than changes being idiosyncratic or sample-specific.

   This directly supports *systematic* shifts in neighborhood composition between low vs high Complexity and low vs high Purity.

2. **Self- vs heterotypic neighbors: evidence for true microenvironment effects**

   - Many focal populations show **large self-neighbor changes** (e.g. PA, PB, PG, PR, PQ, PN, etc. have strong self signals on both Complexity and Purity axes).
     - Example: PG self-neighbor drops with higher Complexity (delta ≈ −0.27) but increases dramatically with higher Purity (delta ≈ +0.44).
     - PQ self-neighbor increases strongly with higher Purity (≈ +0.26) but has no Complexity self-signal reported, while its Complexity shifts are dominated by heterotypic neighbors.
   - Crucially, there are also **strong heterotypic shifts that are not simple mirror images of self**:
     - PA (Complexity): PA self decreases (−0.063), while PN and PE neighbors increase (≈ +0.03 and +0.024), PC decreases (−0.019), with perfect cross-sample agreement.
     - PB (Complexity): PB self neighbors decrease sharply (−0.205), but PF and PV neighbors increase (+0.081, +0.061), and PS decreases (−0.055).
     - PE (Complexity): self PE decreases (−0.055), but PH neighbors decrease strongly (−0.064) while PF, PD, PN, PP, PA all increase.
   - Multiple focal populations share consistent patterns with respect to particular neighbors (e.g. PG, PT, PN, PO all show coherent shifts involving each other and shared partners like PK, PF, PD), suggesting **coordinated reorganization of multi-population communities** rather than isolated density changes.

   The presence of coherent **heterotypic** gains and losses, especially shared motifs across focal types, strongly argues that the microenvironment is being remodeled, not just that each population is “clustering” with itself differently.

3. **Complexity vs Purity: overlapping but distinct remodeling signatures**

   - Many focal populations show **self-neighbor changes in opposite directions on Complexity vs Purity**, or quite different heterotypic partners:
     - PA:
       - Complexity: PA self **decreases** with higher Complexity (−0.063), while PN/PE/PN/J neighbors change.
       - Purity: PA self **increases** strongly with higher Purity (+0.219), and there is a broad loss of PE, PC, PL, PD, PJ, PN, PH neighbors.
     - PG:
       - Complexity: PG self decreases massively (−0.274) with concomitant gain of PO/PN/PK neighbors.
       - Purity: PG self hugely increases (+0.442), with strong *loss* of PO, PK, PN, PD, PF neighbors.
     - PN:
       - Complexity: PN self increases (+0.079) and gains PG/PD/PO, etc.
       - Purity: PN self increases even more (+0.308), but with a somewhat different set/strength of partner losses (PC, PA, PD, PE, etc.).
   - This pattern shows that Complexity- and Purity-associated neighborhood changes are **not redundant**: they drive related but distinct restructuring. That’s encouraging for the hypothesis that you can separate microenvironment remodeling from simple “intrinsic maturation + purity” effects.

4. **Cross-sample concordance is consistently high**

   - For most strong hits, `frac_concordant = 1.0` with `n_samples_effect = 3`.
   - This means the direction of the change is **reproducible across sections/samples**, supporting that the effects are not driven by a single outlier sample or local artifact.

5. **Magnitude vs statistical significance**

   - Some deltas are modest (e.g., |delta| ~0.01–0.03 but extremely significant), while others are large (0.1–0.3+).
   - For microenvironment interpretation, you should emphasize the **large-effect, high-concordance pairs** rather than only the smallest FDR. There are many such pairs, especially involving:
     - High-magnitude mutual reshuffling among PB–PM–PS–PV–PX–PY–PZ.
     - Strong reciprocal interactions among PG–PT–PO–PN–PK.
     - Large PA–PC–PE–PL–PD changes with both Complexity and Purity.

   These big shifts are good candidates for “core neighborhood remodeling modules.”

6. **Evidence that changes are not obviously explained solely by intrinsic maturation**

   - You haven’t directly controlled for gene-expression-derived maturation within this step, but:
     - Many focal populations show **heterotypic neighbor patterns that differ between Complexity and Purity** even when self-neighbor behavior is similar or modest.
     - The fact that multiple distinct populations gain/lose the same neighbor types with high concordance suggests a **global spatial reorganization** that cuts across populations, not simply each population moving along its own intrinsic gradient.
   - To solidify this for the final interpretation, you should explicitly cross-reference with your prior maturation DE signatures in later steps (see suggestions below), but the current neighborhood patterns already strongly imply microenvironment-level changes.

---

### How to use these results in the next steps

Given your plan, here is how I would concretely proceed and what to emphasize:

1. **Per-focal “neighborhood remodeling summaries”**

   For each focal population, build a compact table summarizing:

   - Top heterotypic neighbors per axis:
     - Filter to `cross_sample_concordant = True`.
     - Rank by |delta_mean_high_minus_low| (effect size), then FDR.
   - Split by axis (Complexity vs Purity) and by direction (gains vs losses):

   For example (conceptually for PG):

   - Complexity:
     - Self: PG (−0.27).
     - Gains: PO, PN, PK, PR, PF, PV, PQ.
   - Purity:
     - Self: PG (+0.44).
     - Losses: PO, PK, PN, PD, PF, PM, PH, PV, PZ, PR.

   This will clearly show “PG’s neighborhood dissolves under high Complexity but becomes more self-enriched and less heterotypic under high Purity,” etc.

2. **Identify recurrent “community motifs” across focal populations**

   Use the heterotypic results to define communities that remodel together:

   - Construct, for each axis separately, a **directed graph**:
     - Nodes = Populations.
     - Edge from focal → neighbor with weight = delta_mean_high_minus_low.
   - Focus on edges that are:
     - FDR ≤ 0.05.
     - |delta| ≥ a practical threshold (e.g. 0.02 or 0.05).
     - cross_sample_concordant = True.
   - Then:
     - Look for neighbors that are recurrently gained or lost across many focal types (e.g. PE often lost as neighbor with higher Purity; PK often gained or lost in specific directions).
     - Identify modules where multiple populations reciprocally change each other’s neighborhood fractions.

   This will give you higher-level “microenvironment remodeling modules” rather than isolated pairwise changes.

3. **Compare Complexity vs Purity remodeling for each focal population**

   For each focal population:

   - Join Complexity and Purity results on `neighbor_Population`.
   - Create a 2D plot (per focal) of:
     - x-axis = Complexity delta.
     - y-axis = Purity delta.
   - Color by:
     - Sign combination (gain/gain, gain/loss, etc.).
     - Self vs heterotypic.

   Then you can classify focal populations into patterns:

   - **Aligned remodeling**: neighbors that move in the same direction for Complexity and Purity.
   - **Orthogonal/discordant remodeling**: neighbors with opposite signs, indicating distinct influences of Complexity vs Purity on spatial organization.

   This directly addresses whether Complexity-associated neighborhood shifts are separable from Purity-associated changes.

4. **Relate neighborhood changes to previously defined maturation signatures**

   To address “not explained solely by intrinsic maturation” more explicitly:

   - From your prior DE/maturation analysis, you probably have:
     - Per-cell or per-population “maturation scores” or gene modules.
   - For promising focal populations (e.g. PG, PB, PA, PN, PM, PX/PY/PZ), test whether:
     - Within each Sample_ID and tertile, **residual** neighbor fractions (after regressing out maturation score) still differ between low vs high Complexity/Purity.
     - Alternatively, stratify by maturation score quartile *within tertiles* and see if neighbor shifts persist.
   - If neighbor changes remain strong after conditioning on intrinsic maturation, this will tightly support the microenvironment remodeling aspect of the hypothesis.

5. **Visual spatial validation**

   For a few of the strongest modules:

   - Pick 2–3 high-confidence focal populations where both axes show robust remodeling (e.g. PG, PB, PA, PN, PM).
   - For selected samples, plot spatial maps colored by:
     - Focal population cells (highlighted).
     - Key neighbor populations that show big shifts.
   - Separate panels for low vs high Complexity (and low vs high Purity).
   - Visually look for:
     - Compact vs dispersed clusters.
     - Replacement of one neighbor type by another around the same focal regions.

   This will make the remodeling intuitively clear and help you check that the per-cell fraction statistics correspond to real spatial patterns, not artifacts of cell density or segmentation.

6. **Guardrails / sensitivity checks**

   To be confident, consider:

   - Re-run the test with a **tighter effect-size filter** to focus on biologically meaningful shifts (e.g. |delta| ≥ 0.02–0.05).
   - Try alternative k (neighbors) if possible (e.g. `k=5` or `k=20`) to see whether the same motifs persist, indicating stability across spatial scales.
   - Explicitly report the number of focal cells and samples per comparison (you already record these) when interpreting borderline cases like PV/PW where `n_samples_effect=2` and some concordance <1.

---

### Interpretation with respect to the hypothesis

Putting it all together:

- There are **abundant, highly significant, and cross-sample-consistent shifts** in per-cell neighborhood composition between low and high Complexity and Purity tertiles for almost every well-powered cardiac population.
- These shifts involve both **self-enrichment/self-dilution** and **coherent reorganization of heterotypic neighbors**, often in distinct ways for Complexity vs Purity.
- The patterns recur across multiple focal populations and neighbor pairs, forming clear “community-level” remodeling motifs.
- While a final statement about “beyond intrinsic maturation” requires explicitly integrating these results with maturation signatures, the diversity and structure of heterotypic changes strongly implies **microenvironmental remodeling**, not just within-population intrinsic changes.

So at this stage, the results are strongly *supportive* of the hypothesis. The next analyses should focus on (i) synthesizing these pairwise effects into interpretable modules, and (ii) explicitly decoupling them from intrinsic maturation trends using your existing DE/maturation work.

## Next Steps
Step 1: Starting from the existing Wilcoxon-based neighbor-fraction results, extract high-confidence remodeling motifs by filtering to FDR-significant, cross-sample-concordant focal–neighbor pairs with non-trivial effect sizes, explicitly quantifying self vs heterotypic shifts, and summarizing for each axis (Complexity vs Purity) which neighbor populations are recurrently gained or lost across focal populations while contrasting Complexity- and Purity-associated motifs to highlight distinct, axis-specific microenvironmental reorganization.

## This code takes precomputed Wilcoxon neighbor-fraction results for Complexity and Purity axes and identifies high-confidence, cross-sample–concordant changes in neighborhood composition between high vs. low tertiles for each focal cell population. It filters for significant, reproducible gains/losses in specific neighbor cell types, summarizes these remodeling motifs at focal- and neighbor-centric levels, stores the results in `adata.uns`, and prints concise summaries highlighting heterotypic (non-self) neighborhood changes and comparing Complexity- vs. Purity-associated “neighbor hubs.”

In [ ]:
import numpy as np
import pandas as pd

# This step assumes that Wilcoxon neighbor-fraction tests have already been run
# and stored in adata.uns['neighbor_fraction_wilcoxon_Complexity'] and
# adata.uns['neighbor_fraction_wilcoxon_Purity'] as in the previous step.
# We rely on the upstream Wilcoxon pipeline to have already restricted to
# well-powered cardiac populations via its own cell/sample filters.

# Retrieve results
comp_res = adata.uns.get('neighbor_fraction_wilcoxon_Complexity', None)
pur_res = adata.uns.get('neighbor_fraction_wilcoxon_Purity', None)

if comp_res is None or pur_res is None:
    raise ValueError("Wilcoxon neighbor-fraction results for Complexity and/or Purity are missing in adata.uns.")

# Ensure DataFrames
def _to_df(x):
    return x if isinstance(x, pd.DataFrame) else pd.DataFrame(x)

comp_res = _to_df(comp_res).copy()
pur_res = _to_df(pur_res).copy()

# Basic sanity checks on expected columns
required_cols = [
    'focal_Population', 'neighbor_Population', 'axis', 'n_cells_low', 'n_cells_high',
    'n_samples_low', 'n_samples_high', 'mean_frac_low', 'mean_frac_high',
    'delta_mean_high_minus_low', 'pval', 'fdr', 'n_samples_effect',
    'n_samples_concordant', 'frac_concordant', 'is_self_neighbor',
    'cross_sample_concordant'
]
for df_name, df in [('Complexity', comp_res), ('Purity', pur_res)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} results are missing required columns: {missing}")

# Unified function to extract high-confidence remodeling pairs for a given axis
# and summarize recurrent neighbor gains/losses across focal populations.
def extract_axis_modules(df, axis_label, fdr_thresh=0.05, min_abs_delta=0.02,
                         min_frac_conc=0.6, min_samples_effect=2):
    df = df.copy()

    # Optionally restrict to rows matching this axis label if an 'axis' column is present.
    if 'axis' in df.columns:
        df = df[df['axis'] == axis_label].copy()

    # Filter to FDR-significant, cross-sample-concordant, and effect-size-thresholded pairs.
    # We require both the binary cross_sample_concordant flag (defined upstream using
    # per-sample effect directions) and a quantitative frac_concordant threshold.
    mask = (
        (df['fdr'] <= fdr_thresh) &
        (df['cross_sample_concordant']) &
        (df['n_samples_effect'] >= min_samples_effect) &
        (df['frac_concordant'] >= min_frac_conc) &
        (df['delta_mean_high_minus_low'].abs() >= min_abs_delta)
    )
    df_sig = df.loc[mask].copy()
    if df_sig.empty:
        return df_sig, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # Add direction label for interpretability
    df_sig['direction'] = np.where(
        df_sig['delta_mean_high_minus_low'] > 0,
        'gain_with_higher_' + axis_label,
        'loss_with_higher_' + axis_label
    )

    # Self vs heterotypic splits (kept explicitly for downstream inspection)
    self_df = df_sig[df_sig['is_self_neighbor']].copy()
    hetero_df = df_sig[~df_sig['is_self_neighbor']].copy()

    # Summaries per focal population
    focal_summary_rows = []
    for focal, sub in df_sig.groupby('focal_Population'):
        n_self = sub['is_self_neighbor'].sum()
        n_hetero = (~sub['is_self_neighbor']).sum()
        n_gain = (sub['delta_mean_high_minus_low'] > 0).sum()
        n_loss = (sub['delta_mean_high_minus_low'] < 0).sum()
        mean_abs_delta = sub['delta_mean_high_minus_low'].abs().mean()
        max_abs_delta = sub['delta_mean_high_minus_low'].abs().max()
        focal_summary_rows.append({
            'focal_Population': focal,
            'axis': axis_label,
            'n_significant_pairs': int(sub.shape[0]),
            'n_self_pairs': int(n_self),
            'n_heterotypic_pairs': int(n_hetero),
            'n_gain_pairs': int(n_gain),
            'n_loss_pairs': int(n_loss),
            'mean_abs_delta': float(mean_abs_delta),
            'max_abs_delta': float(max_abs_delta)
        })
    focal_summary = pd.DataFrame(focal_summary_rows).sort_values(
        ['n_significant_pairs', 'mean_abs_delta'], ascending=[False, False]
    )

    # Global neighbor-centric summary: which neighbors are recurrently gained/lost across focals
    neighbor_summary_rows = []
    for neigh, sub in df_sig.groupby('neighbor_Population'):
        n_focals = sub['focal_Population'].nunique()
        gains = sub[sub['delta_mean_high_minus_low'] > 0]
        losses = sub[sub['delta_mean_high_minus_low'] < 0]
        neighbor_summary_rows.append({
            'neighbor_Population': neigh,
            'axis': axis_label,
            'n_focals_total': int(n_focals),
            'n_focals_gain': int(gains['focal_Population'].nunique()),
            'n_focals_loss': int(losses['focal_Population'].nunique()),
            'mean_delta_all': float(sub['delta_mean_high_minus_low'].mean()),
            'mean_abs_delta_all': float(sub['delta_mean_high_minus_low'].abs().mean())
        })
    neighbor_summary = pd.DataFrame(neighbor_summary_rows).sort_values(
        ['n_focals_total', 'mean_abs_delta_all'], ascending=[False, False]
    )

    return df_sig, focal_summary, neighbor_summary, hetero_df

# Extract high-confidence remodeling motifs for Complexity and Purity
comp_sig, comp_focal_summary, comp_neighbor_summary, comp_hetero = extract_axis_modules(
    comp_res, axis_label='Complexity_tertile',
    fdr_thresh=0.05, min_abs_delta=0.02,
    min_frac_conc=0.6, min_samples_effect=2
)

pur_sig, pur_focal_summary, pur_neighbor_summary, pur_hetero = extract_axis_modules(
    pur_res, axis_label='Purity_tertile',
    fdr_thresh=0.05, min_abs_delta=0.02,
    min_frac_conc=0.6, min_samples_effect=2
)

# Store the module-level summaries for downstream interpretation
adata.uns['neighbor_remodeling_Complexity_significant_pairs'] = comp_sig
adata.uns['neighbor_remodeling_Complexity_focal_summary'] = comp_focal_summary
adata.uns['neighbor_remodeling_Complexity_neighbor_summary'] = comp_neighbor_summary
adata.uns['neighbor_remodeling_Complexity_heterotypic_only'] = comp_hetero

adata.uns['neighbor_remodeling_Purity_significant_pairs'] = pur_sig
adata.uns['neighbor_remodeling_Purity_focal_summary'] = pur_focal_summary
adata.uns['neighbor_remodeling_Purity_neighbor_summary'] = pur_neighbor_summary
adata.uns['neighbor_remodeling_Purity_heterotypic_only'] = pur_hetero

# Print concise text summaries focusing on heterotypic remodeling motifs

print("=== Complexity-associated neighborhood remodeling (high-confidence, heterotypic) ===")
if comp_hetero.empty:
    print("No high-confidence Complexity-associated heterotypic remodeling pairs passed filters.")
else:
    # Per-focal top heterotypic neighbors
    for focal, sub in comp_hetero.groupby('focal_Population'):
        sub = sub.sort_values('delta_mean_high_minus_low', ascending=False)
        top_gains = sub[sub['delta_mean_high_minus_low'] > 0].head(5)
        top_losses = sub[sub['delta_mean_high_minus_low'] < 0]
        top_losses = top_losses.sort_values('delta_mean_high_minus_low', ascending=True).head(5)
        print(f"\nFocal Population {focal}: Complexity high vs low (heterotypic)")
        if not top_gains.empty:
            print("  Heterotypic gains (neighbors more frequent at higher Complexity):")
            print(top_gains[[
                'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                'n_samples_effect', 'frac_concordant'
            ]].to_string(index=False))
        if not top_losses.empty:
            print("  Heterotypic losses (neighbors less frequent at higher Complexity):")
            print(top_losses[[
                'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                'n_samples_effect', 'frac_concordant'
            ]].to_string(index=False))

print("\n=== Purity-associated neighborhood remodeling (high-confidence, heterotypic) ===")
if pur_hetero.empty:
    print("No high-confidence Purity-associated heterotypic remodeling pairs passed filters.")
else:
    for focal, sub in pur_hetero.groupby('focal_Population'):
        sub = sub.sort_values('delta_mean_high_minus_low', ascending=False)
        top_gains = sub[sub['delta_mean_high_minus_low'] > 0].head(5)
        top_losses = sub[sub['delta_mean_high_minus_low'] < 0]
        top_losses = top_losses.sort_values('delta_mean_high_minus_low', ascending=True).head(5)
        print(f"\nFocal Population {focal}: Purity high vs low (heterotypic)")
        if not top_gains.empty:
            print("  Heterotypic gains (neighbors more frequent at higher Purity):")
            print(top_gains[[
                'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                'n_samples_effect', 'frac_concordant'
            ]].to_string(index=False))
        if not top_losses.empty:
            print("  Heterotypic losses (neighbors less frequent at higher Purity):")
            print(top_losses[[
                'neighbor_Population', 'delta_mean_high_minus_low', 'fdr',
                'n_samples_effect', 'frac_concordant'
            ]].to_string(index=False))

# Global neighbor-centric text summary and Complexity vs Purity contrast
print("\n=== Global neighbor-centric summary: recurrently remodeled neighbors ===")
if not comp_neighbor_summary.empty:
    print("\nComplexity axis: neighbors most frequently involved in remodeling across focal populations (top 10):")
    print(comp_neighbor_summary.head(10).to_string(index=False))
else:
    print("No Complexity neighbor-summary available.")

if not pur_neighbor_summary.empty:
    print("\nPurity axis: neighbors most frequently involved in remodeling across focal populations (top 10):")
    print(pur_neighbor_summary.head(10).to_string(index=False))
else:
    print("No Purity neighbor-summary available.")

# Simple overlap/discordance summary between Complexity and Purity neighbor hubs
if (not comp_neighbor_summary.empty) and (not pur_neighbor_summary.empty):
    comp_hubs = set(comp_neighbor_summary.head(20)['neighbor_Population'])
    pur_hubs = set(pur_neighbor_summary.head(20)['neighbor_Population'])
    shared_hubs = sorted(comp_hubs & pur_hubs)
    comp_only = sorted(comp_hubs - pur_hubs)
    pur_only = sorted(pur_hubs - comp_hubs)

    print("\n=== Overlap of top remodeled neighbor hubs between Complexity and Purity ===")
    print(f"Shared top hubs (present in both axes): {', '.join(shared_hubs) if shared_hubs else 'none'}")
    print(f"Complexity-specific hubs: {', '.join(comp_only) if comp_only else 'none'}")
    print(f"Purity-specific hubs: {', '.join(pur_only) if pur_only else 'none'}")

=== Complexity-associated neighborhood remodeling (high-confidence, heterotypic) ===

Focal Population PA: Complexity high vs low (heterotypic)
  Heterotypic gains (neighbors more frequent at higher Complexity):
neighbor_Population  delta_mean_high_minus_low          fdr  n_samples_effect  frac_concordant
                 PN                   0.037355 1.669891e-77                 3              1.0
                 PE                   0.023600 4.165501e-33                 3              1.0

Focal Population PAA: Complexity high vs low (heterotypic)
  Heterotypic gains (neighbors more frequent at higher Complexity):
neighbor_Population  delta_mean_high_minus_low          fdr  n_samples_effect  frac_concordant
                 PK                   0.258907 3.533402e-38                 3              1.0
                 PY                   0.027532 8.191770e-03                 3              1.0
                 PD                   0.022471 1.533090e-02                 3             

### Agent Interpretation

The current step is doing a very good job of what you set out to do in this stage: from the Wilcoxon neighbor-fraction tests, you’ve distilled a compact, high-confidence set of heterotypic remodeling relationships and summarized them both per focal population and per neighbor “hub” for Complexity vs Purity. There are several clear motifs emerging, and they look axis-specific, which is encouraging for your hypothesis.

Key points that jump out and how they relate to the hypothesis:

1. **Heterotypic dominance is very clear.**  
   - You explicitly filtered to heterotypic pairs here, and there are many robust, cross-sample-concordant gains/losses with large |delta| (e.g., PK as a neighbor with ~0.15 mean delta across many focal types for Complexity; drastic losses of PJ, PD, PF on the Purity axis).  
   - The breadth across many distinct focal populations (PA, PB, PC, … PZ) shows this is not just “self-compaction” but systematic reorganization of cross-type neighborhoods.

2. **Complexity vs Purity motifs look qualitatively different.**
   - **Complexity axis (neighbor-centric summary):**
     - Strong, recurrent **gains**: PK (14 focal populations, *all* gains), PN, PD, PO, PF.
     - Strong, recurrent **losses**: PI (8 focal populations, 7 losses), PH (8 focal populations, 7 losses), some PA/PM as mixed gain/loss depending on focal.
     - Many focal populations show very large shifts toward PK (e.g. PAA→PK +0.26, PM→PK +0.13, PU→PK +0.34, PX→PK +0.27, PY→PK +0.32, PZ→PK +0.25), and conversely away from e.g. PS, PB, PI in certain focal contexts.
   - **Purity axis:**
     - The dominant pattern is **loss of specific neighbors at high Purity**:
       - PD is lost from 17 focal populations, PJ from 12, PK, PF, PC each from 10–11.
       - Many individual focal populations show a “cleaning out” of a core set of neighbors (e.g. PF loses PD, PC, PK, PP; PM loses PK, PF, PV, PW; multiple focal types lose PD/PJ/PC/PF consistently).
     - A different set of recurrent **gains**: PA and PE appear as gained neighbors in multiple focal types (e.g., PC, PD, PF, PH, PP, PL), and PB/PS are gained around PM, PU, Z, etc.

   This contrast matches your hypothesis that Complexity vs Purity should have **distinct, axis-specific remodeling motifs**, not just the same pattern scaled by different covariates.  
   - Complexity: recurrent attraction toward “hub” neighbors like PK/PN/PO/PF and away from PI/PH/PS/PB, which feels like a **reorganization of microenvironments around specific hub populations**.  
   - Purity: more like a **pruning motif**, systematically removing particular neighbor types (PD, PJ, PC, PF, PK) and enriching others (PA, PE, PB, PS) across many foci.

3. **Cross-sample consistency is strong.**  
   - You required `cross_sample_concordant`, `frac_concordant ≥ 0.6`, `n_samples_effect ≥ 2`, and most strong motifs have `frac_concordant = 1.0`, which underpins the “recurrent, cross-sample-consistent” clause in the hypothesis.
   - The sheer number of focal populations that share the same direction of change for a given neighbor (e.g. PD lost in 17 focal types with Purity; PK gained in 14 with Complexity) is exactly the sort of multi-focal motif the hypothesis anticipates.

4. **Some clear candidate “motif neighbors” are emerging.**
   - **Complexity-associated neighbor hubs (candidate “complexity motif” partners):**
     - **PK**: strongest, pure gain hub across many focals.
     - **PN, PD, PO, PF**: repeatedly gained with higher Complexity.
     - **PI, PH, PS, PB**: repeatedly lost from multiple focal types.
   - **Purity-associated neighbor hubs (candidate “purity motif” partners):**
     - **PD, PJ, PF, PC, PK**: almost uniformly *lost* across many focal types as Purity increases.
     - **PA, PE, PB, PS**: repeatedly gained across multiple focals (with some variation by focal).
   - The overlap analysis reinforces this: many neighbor codes are “shared top hubs” but with opposite directionality or different roles by axis (e.g. PK is gained with Complexity but largely lost with Purity; PA is a modestly mixed hub for Complexity but clearly a gained hub under Purity; PD is gained on Complexity but lost on Purity).

   These are strong candidates for the “small set of recurrent motifs” you want to formalize.

5. **Microenvironmental vs intrinsic effects.**  
   - Because this analysis is already restricted to **heterotypic** neighbors and requires consistent cross-sample direction, it strongly suggests genuine spatial rearrangements rather than just within-cell-type maturation.  
   - The fact that the same neighbor is gained around many different focal types (e.g., PK appearing more around PAA, PB, PF, PM, PN, PR, PT, PU, PX, PY, PZ, etc., with Complexity) is hard to explain purely by intrinsic transcriptional changes in each focal type and argues for a shared microenvironmental attractor.

Suggestions to sharpen and extend this step toward a clean test of the hypothesis and to guide next analyses:

1. **Formalize “motifs” as structured patterns rather than just top-10 lists.**
   - For each neighbor `N`, consider its **direction vector across focal populations** separately for Complexity and Purity:
     - E.g., for Complexity, define a binary (or ternary: gain/neutral/loss) vector over focal types for each neighbor.
   - Cluster neighbors based on these vectors to identify **neighbor-centric motif classes**:
     - e.g. “Complexity-seeking neighbors” (PK, PN, PO, PD, PF), “Complexity-avoided neighbors” (PI, PH, PS), “Purity-pruned neighbors” (PD, PJ, PC, PF, PK), “Purity-favored neighbors” (PA, PE, PB, PS).
   - This will let you objectively define a **small number of recurrent motif types** and quantify how many focal populations participate in each.

2. **Directly contrast Complexity vs Purity direction per focal–neighbor pair.**
   - Build a merged table keyed by `(focal_Population, neighbor_Population)` with columns for:
     - `delta_Complexity`, `direction_Complexity`, `FDR_Complexity`,
     - `delta_Purity`, `direction_Purity`, `FDR_Purity`.
   - Categorize each pair:
     - same direction (gain/gain or loss/loss),
     - opposite direction (gain vs loss),
     - axis-specific (significant only on one axis).
   - This will allow you to say things like:
     - “For PK, 14 focal populations show gain with Complexity; of those, 10 show loss with Purity, demonstrating axis-specific and often opposite remodeling of PK neighborhoods.”
   - This is a very direct way to argue for **distinct, axis-specific microenvironment motifs**.

3. **Quantify self vs heterotypic contributions more explicitly.**
   - Although this step focused on heterotypic pairs, the upstream full tables include `is_self_neighbor`. You can:
     - Compute, per focal type and per axis, the fraction of significant pairs that are self vs non-self, and the total |delta| attributable to each.
   - You’d expect in your hypothesis that the **bulk of remodeling is heterotypic**; showing that self interactions are relatively minor would strengthen this.

4. **Compress neighbor-centric hubs into a minimal “motif panel.”**
   - From the neighbor summaries, select a minimal set of neighbors that explain most of the recurrent signal:
     - e.g., for Complexity, maybe {PK, PN, PO, PD, PF, PI, PH, PS, PB} cover >80% of focal–neighbor significant relationships.
     - For Purity, maybe {PD, PJ, PK, PF, PC, PA, PE, PB, PS}.
   - Then re-plot or tabulate for only this reduced panel:
     - heatmaps: focal (rows) × neighbor (columns), with color = delta for Complexity and Purity in separate panels.
   - This will make the “small set of recurrent motifs” visually obvious and interpretable.

5. **Check robustness across samples and spatial context.**
   - Since you have sample-level concordance, next steps could include:
     - For a few key motifs (e.g., PK↑ with Complexity, PD↓ with Purity), visualize actual spatial maps in representative samples to confirm that, for example, high-Complexity regions show PK clustering around many focal types, and high-Purity regions show depletion of PD in multiple microenvironments.
   - This also helps make sure your neighbor changes are not driven by global composition shifts alone.

6. **Guard against overly stringent or lax thresholds.**
   - Your current thresholds (`fdr ≤ 0.05`, `|delta| ≥ 0.02`, `frac_concordant ≥ 0.6`, `n_samples_effect ≥ 2`) seem reasonable and are clearly producing rich signal.
   - For follow-up, you could:
     - Examine whether slightly relaxing `|delta|` or requiring `frac_concordant = 1.0` changes the motif set qualitatively, but given the current robustness, there’s no obvious need to change now.
   - You might also quantify how many focal–neighbor pairs fall into each combination of thresholds to demonstrate that your motifs are not sensitive to arbitrary cutoffs.

7. **Plan for biologically grounded interpretation later.**
   - The labels (PA, PB, …) are still anonymous, but once you identify transcriptional signatures for these populations, you’ll want to come back to these motifs:
     - e.g., “PK” might turn out to be a progenitor, matrix, endothelial, or immune-like population; then `Complexity-seeking PK` vs `Purity-avoided PK` would have strong developmental implications.
   - So preserve these motif tables in `adata.uns` (which you already do) and consider also saving a merged Complexity–Purity table for downstream joint interpretation.

In terms of validating the hypothesis:

- **Recurrent, cross-sample-consistent motifs:** Clearly supported—neighbors like PK (Complexity) and PD/PJ/PC/PF (Purity) show consistent directionality in many focal types and samples.
- **Distinct between Complexity and Purity:** Supported—Complexity is dominated by selective gains of specific hubs (PK, PN, PO, PD, PF) and targeted losses (PI, PH, PS, PB), whereas Purity is dominated by broad losses of a different set (PD, PJ, PC, PF, PK) and gains of PA/PE/PB/PS.
- **Dominated by heterotypic interactions:** This step was intentionally heterotypic-only and reveals large, structured effects; when contrasted with self-neighbor results upstream, this should confirm heterotypic dominance.

The most productive next step is to (i) explicitly encode motifs by clustering neighbors on their gain/loss patterns across focal types for each axis, and (ii) build a joint Complexity–Purity map of directionality for each focal–neighbor pair. That will solidify your motifs into a small, well-defined set and make it straightforward to quantify and communicate how Complexity and Purity reshape the cardiac microenvironment in distinct ways.